# Multi-channel 3-D convolution on a 3-D grid

Video and medical-imaging models use 3-D convolutions: the filter slides over depth (time or slices), height and width. Implement the forward pass of a multi-channel 3-D convolution layer (cross-correlation, as in deep-learning frameworks) with zero padding and stride:

$$y_{o,d,h,w} = b_o + \sum_{i=0}^{C_{in}-1} \sum_{a=0}^{K_D-1} \sum_{e=0}^{K_H-1} \sum_{f=0}^{K_W-1} W_{o,i,a,e,f}\; x_{i,\ sd + a - p,\ sh + e - p,\ sw + f - p}$$

where $x$ is zero outside its bounds, $s$ is the stride and $p$ the padding (the same on every side and every axis). The output sizes are

$$D_o = \left\lfloor \frac{D + 2p - K_D}{s} \right\rfloor + 1, \quad H_o = \left\lfloor \frac{H + 2p - K_H}{s} \right\rfloor + 1, \quad W_o = \left\lfloor \frac{W + 2p - K_W}{s} \right\rfloor + 1$$

Write `conv3d(x, weight, bias, stride, padding)`.

## The launch: a 3-D grid with a folded axis
The output has **four** indices $(o, d, h, w)$ but a CUDA grid has three axes. Use:
- block `(8, 8, 4)` (256 threads), `w, h, z = cuda.grid(3)` with x → $w$ and y → $h$;
- fold output channel and depth into z: `z = o * D_o + d`, so `o = z // D_o`, `d = z % D_o`;
- grid $(\lceil W_o/8 \rceil, \lceil H_o/8 \rceil, \lceil C_{out} D_o / 4 \rceil)$.

One thread computes one output value, accumulates in a local `float32`, and adds the bias once. Padding is implicit: skip taps that fall outside the input.

## Inputs and output
- `x`: `float32` `(C_in, D, H, W)`; `weight`: `float32` `(C_out, C_in, K_D, K_H, K_W)`; `bias`: `float32` `(C_out,)`.
- `stride` $\ge 1$ and `padding` $\ge 0$ are Python ints; every output size is at least 1.
- Return `y`: `float32` `(C_out, D_o, H_o, W_o)`. Compared with relative tolerance $10^{-4}$, absolute $10^{-5}$.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import numpy as np
from numba import cuda, float32

BLOCK = (8, 8, 4)

def conv3d(x, weight, bias, stride, padding):
    """x: (C_in, D, H, W), weight: (C_out, C_in, KD, KH, KW), bias: (C_out,), all float32.
    Returns float32 (C_out, Do, Ho, Wo)."""
    C_in, D, H, W = x.shape
    C_out, _, KD, KH, KW = weight.shape
    Do, Ho, Wo = D - KD + 1, H - KH + 1, W - KW + 1   # TODO: stride and padding
    y = np.zeros((C_out, Do, Ho, Wo), dtype=np.float32)
    # TODO: launch a kernel on a 3-D grid, folding (C_out, Do) into the z axis
    return y


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[[0.0, 1.0], [2.0, 3.0]], [[4.0, 5.0], [6.0, 7.0]]]], "dtype": "float32"}, {"__nd__": [[[[[1.0, 1.0], [1.0, 1.0]], [[1.0, 1.0], [1.0, 1.0]]]]], "dtype": "float32"}, {"__nd__": [0.5], "dtype": "float32"}, 1, 0], "e": {"__nd__": [[[[28.5]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]]]], "dtype": "float32"}, {"__nd__": [[[[[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]]]], [[[[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]]]]], "dtype": "float32"}, {"__nd__": [0.0, -1.0], "dtype": "float32"}, 1, 1], "e": {"__nd__": [[[[8.0, 12.0, 8.0], [12.0, 18.0, 12.0], [8.0, 12.0, 8.0]], [[12.0, 18.0, 12.0], [18.0, 27.0, 18.0], [12.0, 18.0, 12.0]], [[8.0, 12.0, 8.0], [12.0, 18.0, 12.0], [8.0, 12.0, 8.0]]], [[[7.0, 11.0, 7.0], [11.0, 17.0, 11.0], [7.0, 11.0, 7.0]], [[11.0, 17.0, 11.0], [17.0, 26.0, 17.0], [11.0, 17.0, 11.0]], [[7.0, 11.0, 7.0], [11.0, 17.0, 11.0], [7.0, 11.0, 7.0]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[0.10468166321516037, -0.8065453171730042, 0.8919731378555298, -0.9285507798194885, -1.1356676816940308, 0.5134580135345459], [1.319050669670105, 1.2758177518844604, 1.3556002378463745, 0.40296003222465515, -0.34895575046539307, 0.5779352188110352], [0.5794493556022644, 0.0661078616976738, -0.6986229419708252, -0.17757895588874817, 0.26550811529159546, -1.155428171157837], [-0.36372479796409607, -1.2169758081436157, 1.7756662368774414, 0.7107280492782593, -0.14065319299697876, 0.9527892470359802], [0.17559249699115753, -0.5476267337799072, 0.028777964413166046, -0.1313570886850357, 3.641681671142578, -0.9858279228210449]], [[-2.7848196029663086, 0.7665310502052307, -1.3263137340545654, -1.3989546298980713, 1.3564752340316772, -0.8927116394042969], [-1.3230607509613037, 0.01835167594254017, -1.2653939723968506, 0.7417453527450562, -0.7936019897460938, 0.09403914213180542], [1.506269931793213, 2.020174980163574, -2.2683587074279785, -0.5005839467048645, -0.4062609076499939, 0.8529298305511475], [-0.43904149532318115, -0.9312676787376404, 1.5131419897079468, 1.3407573699951172, 0.13185586035251617, -2.1364974975585938], [-0.14161764085292816, -1.4062128067016602, 2.0727298259735107, 0.5134504437446594, 0.3769960105419159, 0.8980483412742615]], [[0.7343261241912842, -1.2340610027313232, 0.634585976600647, 0.3554089367389679, -2.610100746154785, 1.208814263343811], [0.4471503794193268, -0.0946047306060791, 0.5522430539131165, -0.8403838872909546, 1.0841730833053589, -0.3856477737426758], [-1.293161392211914, 2.119006872177124, 0.8554474115371704, 0.08914987742900848, 0.30090832710266113, 0.5626627802848816], [1.538124680519104, 0.7753117084503174, 1.4379879236221313, -0.6563751697540283, -0.7444276213645935, 0.25438451766967773], [0.4786055386066437, -0.028811965137720108, -0.8431254625320435, -0.2960953414440155, 0.9279054403305054, 0.8590631484985352]], [[-1.1776049137115479, 0.8619683980941772, 0.49836277961730957, 0.40042442083358765, 0.9738825559616089, 0.3124775290489197], [1.8074710369110107, -0.491681307554245, -0.9381097555160522, 0.3162057101726532, 0.498546838760376, -1.062974452972412], [0.5288833379745483, 0.3239125907421112, -1.1330124139785767, -0.29039397835731506, 1.1363911628723145, 1.0072283744812012], [-1.0945528745651245, 0.8100159168243408, 0.08211134374141693, 2.198730707168579, -0.18404521048069, 0.66422039270401], [-1.5809545516967773, 1.1641223430633545, -0.015859873965382576, -0.548403799533844, -0.49963244795799255, -0.14341841638088226]]], [[[-0.573279857635498, 0.6214244961738586, -0.4837692379951477, -0.2837143838405609, 3.0221776962280273, -0.045127883553504944], [0.511593759059906, 0.31080323457717896, -0.05607115849852562, 0.1421217918395996, -1.3477555513381958, 0.7503446936607361], [0.2481265515089035, 0.1531074196100235, -1.1415971517562866, 0.9608229994773865, -1.4083664417266846, -0.09069685637950897], [-0.7007060050964355, -1.4378266334533691, 1.5623375177383423, 1.4338330030441284, 1.2869844436645508, -0.006138356868177652], [-0.8896258473396301, 0.1835116595029831, -0.14141488075256348, -1.8243428468704224, 1.2042388916015625, 0.8366813659667969]], [[-0.6325790882110596, -1.0399388074874878, -0.07576574385166168, 0.21080642938613892, 2.196359634399414, 0.4142724275588989], [-1.0330719947814941, 0.6605967879295349, -1.2315905094146729, -1.052594780921936, 0.5111566781997681, -0.22079677879810333], [-1.4502513408660889, -0.7081471085548401, -0.48418334126472473, -1.0513579845428467, -0.4927869439125061, 0.1285572350025177], [-0.2700670063495636, -0.880038321018219, 0.7451338171958923, 1.5828863382339478, 0.8827695846557617, -1.0951634645462036], [-0.8214902281761169, 0.09469897300004959, 0.1831129938364029, -0.9538848400115967, 0.48369696736335754, 0.021739980205893517]], [[-1.2312356233596802, -1.772970199584961, 0.005742082372307777, -0.6701810359954834, 1.9603848457336426, -0.949763834476471], [-0.5993463397026062, 2.8347551822662354, -1.178462266921997, 1.0377352237701416, 0.19423553347587585, -0.29859885573387146], [-1.081479787826538, 0.14411364495754242, 1.579131841659546, 0.8922150135040283, 0.007472758647054434, 0.9403536319732666], [-0.35181954503059387, 0.20487911999225616, 1.2150535583496094, 0.09418055415153503, 0.8932214975357056, 0.14734315872192383], [-1.050977110862732, 1.3456121683120728, -0.7042168378829956, 1.3721213340759277, 1.1029667854309082, 1.4536974430084229]], [[0.8066694140434265, -0.15114876627922058, -0.902579665184021, -0.5112901926040649, -0.5131332874298096, 1.0701137781143188], [-1.655621886253357, 0.998753011226654, -0.2515871524810791, 0.46067750453948975, 0.9378744959831238, 0.6399665474891663], [1.0816839933395386, -0.6805874705314636, -0.42184802889823914, -1.3269211053848267, -0.19838397204875946, -0.11144847422838211], [0.41362035274505615, 0.6052985191345215, 1.241477131843567, 1.3237777948379517, -0.5707613825798035, -0.18363605439662933], [-0.38186416029930115, -0.28560498356819153, 1.1403928995132446, 1.5626875162124634, 0.7494915723800659, 0.5809842348098755]]]], "dtype": "float32"}, {"__nd__": [[[[[-0.8456148505210876]]], [[[-0.5039567351341248]]]], [[[[-0.08726052194833755]]], [[[-2.330630302429199]]]], [[[[0.4164133667945862]]], [[[-0.15427318215370178]]]]], "dtype": "float32"}, {"__nd__": [1.8573768138885498, -0.11370290070772171, -1.6718770265579224], "dtype": "float32"}, 1, 0], "e": {"__nd__": [[[[2.057764768600464, 2.2262325286865234, 1.3469098806381226, 2.785552978515625, 1.2946674823760986, 1.4459316730499268], [0.48414695262908936, 0.6218950748443604, 0.7393186092376709, 1.4450045824050903, 2.831669330596924, 0.99052494764328], [1.2423408031463623, 1.7243155241012573, 3.023458480834961, 1.5233269929885864, 2.3426151275634766, 2.880131244659424], [2.518073558807373, 3.611072063446045, -0.4315035343170166, 0.5337848663330078, 1.3277307748794556, 1.0547776222229004], [2.157226085662842, 2.227976083755493, 1.9043086767196655, 2.8878440856933594, -1.828967571258545, 2.2693562507629395]], [[4.531054496765137, 1.7332708835601807, 3.0171098709106445, 2.9341163635253906, -0.3965489864349365, 2.403491497039795], [3.496800184249878, 1.508946180343628, 3.548081159591675, 1.760608196258545, 2.270857572555542, 1.8891279697418213], [1.314516544342041, 0.5059623718261719, 4.0195417404174805, 2.810516834259033, 2.4492602348327637, 1.0713393688201904], [2.364738941192627, 3.088371753692627, 0.20232629776000977, -0.07409381866455078, 1.3009998798370361, 4.215945720672607], [2.3911263942718506, 2.998767137527466, 0.012364745140075684, 1.9039121866226196, 1.29482102394104, 1.0870177745819092]], [[1.8569092750549316, 3.794417381286621, 1.3178677558898926, 1.8945800065994263, 3.0765676498413086, 1.3138253688812256], [1.7813044786453247, 0.5087820291519165, 1.984285831451416, 2.045044183731079, 0.8426976203918457, 2.3339672088623047], [3.4959123134613037, -0.007113933563232422, 0.3381836414337158, 1.332352638244629, 1.5991582870483398, 0.907683253288269], [0.7340176105499268, 1.0985114574432373, 0.029058456420898438, 2.3649544715881348, 2.036731004714966, 1.568010926246643], [1.9823079109191895, 1.2036103010177612, 2.9252309799194336, 1.4162696599960327, 0.516878604888916, 0.3983396291732788]], [[2.446650505065918, 1.2046560049057007, 1.890815019607544, 1.776440143585205, 1.2924442291259766, 1.0538501739501953], [1.1633142232894897, 1.7698215246200562, 2.7774453163146973, 1.3578269481658936, 0.9631500244140625, 2.4337282180786133], [0.8650233149528503, 1.9264581203460693, 3.028062105178833, 2.771649122238159, 0.9964045286178589, 1.061814785003662], [2.574500322341919, 0.8673710823059082, 1.1622915267944336, -0.6690292358398438, 2.300647258758545, 1.388246774673462], [3.3866984844207764, 1.016910195350647, 1.296079397201538, 1.5335882902145386, 1.9021620750427246, 1.68586266040802]]], [[[1.2132660150527954, -1.4916340112686157, 0.9359503984451294, 0.6285563111305237, -7.058182716369629, -0.053331103175878525], [-1.4211399555206299, -0.9493988752365112, -0.10131214559078217, -0.48009875416755676, 3.0578670501708984, -1.9129098653793335], [-0.742557168006897, -0.47630828619003296, 2.6079001426696777, -2.3375306129455566, 3.145510196685791, 0.19850122928619385], [1.5511225461959839, 3.3435332775115967, -3.909879684448242, -3.5174560546875, -3.100914478302002, -0.18253755569458008], [1.9443639516830444, -0.49361452460289, 0.21337175369262695, 4.149628162384033, -3.2381136417388916, -1.9776740074157715]], [[1.6036099195480347, 2.243122100830078, 0.17861387133598328, -0.48294124007225037, -5.350971698760986, -1.001320242881775], [2.409456729888916, -1.6549111604690552, 2.867098093032837, 2.2747812271118164, -1.2357699871063232, 0.3926869034767151], [3.1348588466644287, 1.3604446649551392, 1.2126877307891846, 2.3803048133850098, 1.0702518224716187, -0.4877493679523468], [0.5540344715118408, 2.018603801727295, -1.9823719263076782, -3.91982102394104, -2.1826183795928955, 2.625149965286255], [1.8132447004318237, -0.2117043435573578, -0.7213391065597534, 2.064646005630493, -1.273918628692627, -0.2427349090576172]], [[2.691774368286133, 4.126120090484619, -0.18245987594127655, 1.4172282218933105, -4.45487642288208, 1.9943636655807495], [1.2441332340240479, -6.712213516235352, 2.584667921066284, -2.4589478969573975, -0.6609996557235718, 0.6158725023269653], [2.5196683406829834, -0.6344841718673706, -3.8687222003936768, -2.2009057998657227, -0.15737655758857727, -2.3544180393218994], [0.572040855884552, -0.6588544845581055, -3.0710232257843018, -0.2759273052215576, -2.1305129528045654, -0.4793030619621277], [2.2939727306365967, -3.2473134994506836, 1.6011377573013306, -3.2857730388641357, -2.7652804851531982, -3.5766966342926025]], [[-1.8909926414489746, 0.16335317492485046, 1.9463893175125122, 1.0429843664169312, 0.9972397089004517, -2.635009527206421], [3.587218761444092, -2.3985226154327393, 0.554513692855835, -1.2149641513824463, -2.343045234680176, -1.5124726295471191], [-2.680859088897705, 1.444230079650879, 0.9683362245559692, 3.004199266433716, 0.249494731426239, 0.0581509992480278], [-0.9821877479553223, -1.5951123237609863, -3.0142924785614014, -3.3908019065856934, 1.2325907945632935, 0.2563246488571167], [0.9142361879348755, 0.45035481452941895, -2.770153284072876, -3.7078959941864014, -1.8168925046920776, -1.4552476406097412]]], [[[-1.5398445129394531, -2.103602409362793, -1.2258148193359375, -2.014768600463867, -2.611025333404541, -1.4511041641235352], [-1.2015318870544434, -1.1885581016540527, -1.0987366437911987, -1.526004672050476, -1.6092643737792969, -1.5469751358032227], [-1.4688658714294434, -1.6679692268371582, -1.7866750955581665, -1.894052505493164, -1.3440427780151367, -2.1390206813812256], [-1.715236783027649, -1.9568239450454712, -1.173492670059204, -1.597122311592102, -1.928994059562683, -1.2741758823394775], [-1.461512565612793, -1.9282270669937134, -1.6380770206451416, -1.4451286792755127, -0.34121382236480713, -2.2114663124084473]], [[-2.7339231967926025, -1.1922485828399658, -2.2124831676483154, -2.28694224357605, -1.4458619356155396, -2.10752534866333], [-2.063441753387451, -1.766147494316101, -2.008802652359009, -1.2006171941757202, -2.0812013149261475, -1.598654866218567], [-0.820911169052124, -0.721401035785675, -2.541755437850952, -1.7181305885314941, -1.7650257349014282, -1.336538553237915], [-1.8130357265472412, -1.923902988433838, -1.1567386388778687, -1.3577646017074585, -1.7531580924987793, -2.3925888538360596], [-1.6041145324707031, -2.27205228805542, -0.8370140790939331, -1.3109105825424194, -1.5895123481750488, -1.3012715578079224]], [[-1.176147222518921, -1.9122347831726074, -1.408512830734253, -1.4204890727996826, -3.061192512512207, -1.0219875574111938], [-1.3932145833969116, -2.1485984325408936, -1.2601104974746704, -2.1819188594818115, -1.250378131866455, -1.7864000797271729], [-2.0435233116149902, -0.8117271065711975, -1.5592750310897827, -1.7723987102508545, -1.5477275848388672, -1.5826480388641357], [-0.9771050214767456, -1.3806341886520386, -1.2605297565460205, -1.9597299098968506, -2.119666814804077, -1.5886789560317993], [-1.3104417324066162, -1.8914666175842285, -1.9143239259719849, -2.006856679916382, -1.4556429386138916, -1.538418173789978]], [[-2.2866950035095215, -1.2896236181259155, -1.325108289718628, -1.426256537437439, -1.1871765851974487, -1.706847071647644], [-0.6638039350509644, -2.030700445175171, -2.023705244064331, -1.6112749576568604, -1.6089643239974976, -2.2132434844970703], [-1.6185177564620972, -1.4319990873336792, -2.078598737716675, -1.588092565536499, -1.1680631637573242, -1.2352601289749146], [-2.191473960876465, -1.4279569387435913, -1.8292113542556763, -0.960519552230835, -1.660462737083435, -1.3669567108154297], [-2.271296262741089, -1.1430597305297852, -1.8544132709503174, -2.1413204669952393, -1.9955570697784424, -1.8212286233901978]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[0.14722216129302979, -0.1335310935974121, 0.28642329573631287, -1.1096338033676147, 0.05606847256422043, -1.7908542156219482], [0.39611345529556274, -0.14989669620990753, -2.2803027629852295, 0.7358688116073608, 1.1031534671783447, 1.1837184429168701], [0.7322801351547241, -0.5860660672187805, -0.7147589921951294, 0.5303341150283813, 1.1479816436767578, 2.82061505317688], [0.3552108407020569, -1.7482285499572754, -0.3092803955078125, 0.8671424984931946, 0.6636236906051636, 1.0781539678573608], [-0.8887432813644409, -0.4095929265022278, 0.7181432843208313, 0.4380524754524231, 0.417124480009079, -0.5995995998382568], [-0.47020837664604187, 0.5284940004348755, 0.9218194484710693, -1.2731924057006836, 0.2241811901330948, 0.602205753326416]], [[0.3198188245296478, 0.35104963183403015, -0.07710755616426468, -0.7346887588500977, 1.1738203763961792, -1.276419997215271], [0.29217827320098877, -1.1644463539123535, -0.6373396515846252, -0.14950720965862274, -1.0626459121704102, 0.070877306163311], [0.8445547223091125, 1.1845312118530273, -0.4643956124782562, 0.2943641245365143, -0.5066274404525757, -1.1367496252059937], [-0.5947785377502441, 2.014265298843384, -1.052378535270691, -0.6117310523986816, -0.36774393916130066, 0.9766630530357361], [0.10409028083086014, -0.5775114893913269, -0.6425269842147827, 0.2063571959733963, 0.2009143978357315, -1.8163305521011353], [0.06826216727495193, 0.1507311463356018, 0.3411676287651062, -0.3959938883781433, 0.47215306758880615, -1.1700217723846436]], [[0.20828677713871002, 2.767233371734619, 0.42802467942237854, -0.9328165650367737, 0.6026407480239868, 0.9940878748893738], [0.7562634348869324, -0.3287214934825897, 0.12024721503257751, 0.2932545244693756, 3.2393689155578613, -1.8991639614105225], [-0.3693033456802368, -0.2717113792896271, -0.33433184027671814, 0.08749683946371078, 1.1389076709747314, -1.253661870956421], [-0.18577660620212555, -0.986648440361023, 0.6215872764587402, 1.8750722408294678, -1.0035649538040161, -0.1501755714416504], [-0.38370102643966675, -0.9633074998855591, -0.4386160373687744, -0.060737013816833496, -1.2880172729492188, 0.7598475217819214], [-0.9382107257843018, -2.1720163822174072, 0.11253824084997177, -0.25931864976882935, 0.6896458268165588, -0.03928720951080322]], [[0.30010396242141724, 0.533517062664032, -1.2597575187683105, 1.5364747047424316, 0.9192735552787781, 1.2106959819793701], [-0.7380468249320984, 0.08760391175746918, 0.19852450489997864, -1.3605144023895264, 0.590932309627533, 0.4972577393054962], [0.5038295388221741, 0.2191881388425827, 1.0774073600769043, 0.5104904770851135, 0.36230775713920593, 0.8126958608627319], [-0.19997847080230713, -0.38948842883110046, 0.8263952136039734, -0.10470566898584366, 0.3211789131164551, -1.4646482467651367], [0.25281956791877747, -1.7518723011016846, -0.6221041679382324, -0.19581711292266846, 2.4262242317199707, -0.2566211223602295], [0.7582154273986816, -0.5624875426292419, -0.307936429977417, -1.6442822217941284, 1.0800663232803345, 0.5630863308906555]], [[0.7937012910842896, 0.38054803013801575, -0.07341064512729645, 0.6159693002700806, -0.06521376222372055, 1.4636621475219727], [0.567226767539978, -0.352751761674881, -1.1573970317840576, -0.18216250836849213, -1.1056807041168213, 0.4299522638320923], [-0.07396101951599121, -1.661865234375, 1.1680203676223755, -0.6310774087905884, -1.8840402364730835, 1.3508014678955078], [-0.3295409083366394, -0.5509608387947083, -0.5539348125457764, 0.07061620056629181, -0.09335260838270187, -1.2369695901870728], [0.009768083691596985, 0.013084993697702885, 0.3951994776725769, -1.6142821311950684, -0.3886035084724426, -1.4030207395553589], [-0.6307106018066406, -1.2179241180419922, 0.37708139419555664, -0.14228500425815582, -0.15750755369663239, 0.17005737125873566]], [[-0.9108189940452576, -1.3301247358322144, -1.234539270401001, 1.0696539878845215, -0.32135218381881714, 2.4105823040008545], [0.7905035018920898, 0.08174345642328262, 0.8690222501754761, -0.6824533343315125, 0.19018012285232544, -0.035041049122810364], [0.9404796957969666, -1.5313310623168945, -0.9750115275382996, 0.511040449142456, 0.8501442074775696, 1.540849208831787], [-0.050916798412799835, 0.13137896358966827, -1.3467803001403809, 0.5204359889030457, 0.6325441002845764, -0.25094446539878845], [0.34878888726234436, 0.21161293983459473, 0.48594745993614197, 0.10489361733198166, 1.4968355894088745, 2.092280864715576], [-0.15676814317703247, -0.10533984750509262, -1.2755727767944336, -0.029639912769198418, 1.2838635444641113, -0.9713876247406006]]]], "dtype": "float32"}, {"__nd__": [[[[[-0.781018078327179, 1.674926519393921, -1.7482104301452637], [1.105424165725708, 0.6243992447853088, 0.030694516375660896], [-0.6137710213661194, 0.30934861302375793, 1.3846690654754639]], [[-1.2575045824050903, 1.635642409324646, 0.935398280620575], [-0.5211971998214722, 0.008182935416698456, 0.9037092924118042], [-1.2529857158660889, -1.5505398511886597, -1.1246784925460815]], [[-1.1535143852233887, 0.11952333897352219, 0.9256125092506409], [-0.8046424984931946, -1.4331399202346802, 0.051618561148643494], [-0.7243105173110962, -0.3297517001628876, 0.15611572563648224]]]]], "dtype": "float32"}, {"__nd__": [0.5881834626197815], "dtype": "float32"}, 2, 1], "e": {"__nd__": [[[[-0.6952483057975769, 3.373596668243408, -5.102459907531738], [0.3061181902885437, -0.1422482132911682, 2.7867748737335205], [0.9065317511558533, 1.1446683406829834, 2.361772298812866]], [[0.8307309746742249, -0.055962979793548584, -3.1568777561187744], [7.6479573249816895, -4.460455894470215, 2.7024145126342773], [-3.0302088260650635, 7.041321277618408, -9.92945671081543]], [[1.541320562362671, 3.8039166927337646, 7.214831829071045], [-2.074446678161621, 7.130455493927002, 1.1294682025909424], [1.460993766784668, -2.606224536895752, 2.152869462966919]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[0.23817560076713562, 0.3914220333099365, -0.07532403618097305, 1.2948793172836304, -0.8720994591712952, 2.9856691360473633, 0.07972465455532074], [0.6434237957000732, 0.35276010632514954, -1.1015760898590088, 0.15129421651363373, 0.9535216093063354, 1.0080902576446533, 0.6045935750007629], [-0.20919518172740936, -0.6299582719802856, -1.0289759635925293, 1.3683127164840698, 1.5899829864501953, -0.4153262674808502, 0.4414571225643158], [-0.165479376912117, -0.014661715365946293, -0.2579784691333771, 1.2471941709518433, 0.2254936248064041, -0.3808791935443878, 1.2829291820526123]], [[-1.388672947883606, 0.5295708179473877, 0.2717956304550171, -0.9577822685241699, -1.7737113237380981, -1.6600924730300903, -0.47839444875717163], [-0.755990743637085, -2.0770208835601807, -0.3052842319011688, -0.614726185798645, -0.31076890230178833, -0.3798964321613312, -0.42206257581710815], [-0.8086985349655151, 0.23742814362049103, -0.3201155960559845, 1.3099899291992188, 0.0881139487028122, -1.1557451486587524, 2.1612210273742676], [0.3317468464374542, 0.9178056716918945, -0.05774283781647682, -1.2706671953201294, 0.6617989540100098, 0.4929293692111969, -1.7829803228378296]], [[-1.0635671615600586, 0.7350525856018066, -0.5688027143478394, -0.39177191257476807, 0.6036184430122375, 2.1529290676116943, 0.16488562524318695], [1.7414696216583252, -1.2160755395889282, 0.8843459486961365, -0.008245948702096939, -2.1010775566101074, -0.08117183297872543, -0.645389199256897], [-0.6562961339950562, 2.7867233753204346, -2.1319637298583984, 0.13287299871444702, -2.0905141830444336, 0.37674590945243835, -0.17339301109313965], [0.9877891540527344, 0.38396990299224854, 0.10706045478582382, -0.6918378472328186, -0.35605019330978394, 1.2406010627746582, 0.89149409532547]], [[-1.1277912855148315, -0.23854972422122955, 1.0470967292785645, 0.8922582864761353, 2.063248872756958, -0.8073171377182007, 0.6653221845626831], [-0.2514828145503998, 0.5270697474479675, -0.0007556338096037507, 0.8059741854667664, 1.2608410120010376, -0.7005738019943237, -2.5143799781799316], [0.5987846851348877, 0.03030473180115223, -0.48349907994270325, 0.8494659662246704, 2.3144137859344482, 0.4772166609764099, -0.7221522331237793], [-0.320161372423172, -0.48822781443595886, -0.8037633895874023, -0.7024321556091309, -0.5984635353088379, -0.5508480072021484, 0.20769049227237701]], [[-0.24374067783355713, 0.12761612236499786, -1.1320624351501465, -0.8625273704528809, 0.7456511855125427, -1.6820935010910034, -0.023605255410075188], [-0.3253891170024872, -0.8326120972633362, 0.04418593645095825, -0.25519657135009766, -1.3629497289657593, 1.0088375806808472, 0.7087444067001343], [-2.056272029876709, 0.10337892919778824, 1.1063052415847778, 2.3170523643493652, 1.2930908203125, -1.5029407739639282, -0.5223432183265686], [-0.6313387155532837, -0.9182599186897278, -1.6317548751831055, 0.10928165912628174, 0.16366131603717804, -0.487255722284317, 0.4165632426738739]]], [[[-0.3985348343849182, 1.1430633068084717, 0.6614748239517212, 0.6557782292366028, -0.4421502649784088, -2.037411689758301, -0.37099966406822205], [0.2739964723587036, -2.992950916290283, -0.42882636189460754, 1.9277065992355347, -0.9080404043197632, -0.44982272386550903, 0.9932695627212524], [1.6541045904159546, -0.530976414680481, 1.7587004899978638, -0.0757729709148407, -0.02658357284963131, -0.7783015370368958, -0.013773570768535137], [1.2538964748382568, -1.5252680778503418, -0.3134147822856903, -1.5181059837341309, -0.4879855513572693, 0.04058803990483284, 0.18162228167057037]], [[-0.5428044199943542, 0.40202847123146057, -0.29835018515586853, 0.9574355483055115, -1.6955903768539429, -1.704715609550476, 1.8357759714126587], [-1.3781814575195312, -2.7330470085144043, -1.6292424201965332, 0.0033624055795371532, 0.468802809715271, 2.0839829444885254, 2.136002779006958], [-0.2260080724954605, 1.2487226724624634, 0.22215671837329865, 0.20735014975070953, 1.0473995208740234, 1.3388820886611938, -0.18966270983219147], [1.3063095808029175, 0.5927284359931946, 0.22325055301189423, -0.4908052980899811, 0.5704984664916992, 1.1109225749969482, 0.5712917447090149]], [[-0.048134855926036835, 1.0400283336639404, 0.46682208776474, -0.8751986026763916, 1.275557041168213, -2.723231077194214, -1.1791741847991943], [-1.605234980583191, 0.5422102808952332, 0.18843288719654083, 0.15991508960723877, 0.06183980777859688, -1.5931495428085327, -0.6932857036590576], [1.3097752332687378, 0.5446407794952393, -0.5921136140823364, -1.7616158723831177, 1.053966760635376, 0.3072921335697174, 0.4785695970058441], [0.25150731205940247, 0.5633454322814941, -3.1280057430267334, 0.4177548587322235, 0.7365785241127014, -0.9565303325653076, 1.8995046615600586]], [[-0.5496648550033569, -0.7642958760261536, 0.5289381742477417, -0.5174562335014343, -0.9957733154296875, -0.4552604854106903, -0.6968389749526978], [-0.524337649345398, -0.07812348008155823, 0.25273552536964417, -1.620972752571106, 0.5061360597610474, 0.37423449754714966, 1.343759536743164], [0.9135084748268127, -0.6850994825363159, -0.4764286279678345, -0.2838996350765228, -0.1346769779920578, -0.5129191279411316, 0.51651930809021], [-0.04702385514974594, 0.24304139614105225, -1.336513876914978, -2.088684320449829, 0.1433308720588684, 0.05988086760044098, -1.66446852684021]], [[-0.6378152370452881, -0.6708289980888367, 0.04170143976807594, -0.16931498050689697, 0.2830716669559479, -1.0673376321792603, -0.3382124602794647], [-0.4455571174621582, 0.27410003542900085, 0.3906048536300659, -0.2802351415157318, -1.7583972215652466, -0.20932427048683167, -1.6525171995162964], [-0.8883363604545593, -0.38854801654815674, -0.3856944441795349, 2.2428956031799316, 1.444159746170044, 1.3956067562103271, 0.7098613381385803], [-1.0122216939926147, 0.7245346307754517, -0.49633216857910156, 0.07886707037687302, -1.4859774112701416, -0.42789337038993835, 1.1303259134292603]]], [[[-2.3149425983428955, 0.6776478290557861, -0.49064385890960693, -0.7105449438095093, 1.3481594324111938, -0.6676077246665955, 0.8159152865409851], [0.3869756758213043, -0.25420981645584106, 0.3990110754966736, -0.5847289562225342, 1.404101848602295, 0.9579253792762756, -1.4980522394180298], [-0.6298633217811584, -0.03554855287075043, 0.07435876131057739, 0.9187221527099609, -0.983961820602417, -1.3671116828918457, 0.06673331558704376], [-0.3153831958770752, 0.6341211199760437, 1.670454502105713, -0.8329193592071533, -1.049493670463562, -1.1873140335083008, 1.3047987222671509]], [[0.8324283957481384, -0.1992729753255844, 0.37046656012535095, -1.2783170938491821, 0.6288012266159058, -0.26093393564224243, -0.30345895886421204], [0.03385864570736885, 0.1680319756269455, 0.08642609417438507, -2.964627742767334, 1.1087850332260132, -1.5251621007919312, -0.8398411273956299], [0.23912771046161652, -0.785703182220459, -1.0574543476104736, 0.10469824075698853, 0.7376207113265991, -1.1420300006866455, -0.2935839295387268], [-1.6992006301879883, -0.05259154736995697, 1.97209632396698, 0.9478409290313721, -0.6189083456993103, -0.018977073952555656, 0.090033158659935]], [[0.03775369003415108, 0.38415250182151794, 1.0024759769439697, 0.43446943163871765, -0.040877778083086014, 0.4497784376144409, -0.46757277846336365], [-0.7677901387214661, -1.5298657417297363, 0.5973014831542969, 0.32321590185165405, 0.5609661340713501, -0.4103582203388214, 1.0977182388305664], [0.9290080070495605, 0.2641947567462921, 1.082461953163147, -0.023888874799013138, 0.7961979508399963, -0.7250837683677673, 0.932643473148346], [-1.464530110359192, -0.8801568746566772, 0.04014096036553383, 1.0927095413208008, 0.049299608916044235, -1.1128580570220947, 0.4755770266056061]], [[-2.427096366882324, 0.012472515925765038, 0.08550367504358292, -0.12054150551557541, 0.36305972933769226, 0.3028390407562256, 0.03588494658470154], [-0.6738379001617432, -0.435993492603302, 0.38722753524780273, -1.360336422920227, -1.1447724103927612, -0.04827500507235527, -0.22447921335697174], [0.6594240069389343, 0.8158895373344421, -0.21176323294639587, -0.6853511929512024, 0.8659071922302246, -1.602799654006958, 0.8035513162612915], [1.7187212705612183, -0.4763312041759491, 0.7732482552528381, 0.500115156173706, -0.6317205429077148, -0.6671774387359619, 0.9716262817382812]], [[-0.12070556730031967, -0.8946854472160339, -2.2892532348632812, -0.26289376616477966, 0.007130964659154415, 0.4367397725582123, -0.10163036733865738], [-1.454555869102478, -1.333044409751892, -0.7876719832420349, 0.05502142757177353, -0.6648480296134949, 0.28495439887046814, 0.6659325361251831], [0.29802194237709045, 0.7307451367378235, 0.46112605929374695, -0.2974490225315094, -0.7389004826545715, 1.502607822418213, 0.4180169999599457], [1.0462473630905151, 1.1345136165618896, 0.39014744758605957, 0.3530483841896057, 1.0083993673324585, -0.43383505940437317, -1.1875985860824585]]]], "dtype": "float32"}, {"__nd__": [[[[[0.5542805194854736, 1.1033490896224976], [-0.6157613396644592, -0.27821290493011475], [0.4104699194431305, -0.859018862247467]], [[-1.492993712425232, 0.2810693085193634], [-0.18030640482902527, -1.5153864622116089], [1.0312395095825195, -0.9521855711936951]]], [[[-1.4516063928604126, 0.0591321736574173], [0.06602871417999268, 1.842269778251648], [0.4187620282173157, -1.163244605064392]], [[-0.9722167253494263, -0.8067201375961304], [0.39658859372138977, -0.5356863141059875], [0.6440862417221069, -0.40668532252311707]]], [[[-1.8463362455368042, -2.3479154109954834], [-1.1148953437805176, -0.6044748425483704], [-0.39346733689308167, 0.7349743843078613]], [[0.5994810461997986, -1.1183840036392212], [0.24279440939426422, -1.767518162727356], [0.9393317699432373, -0.1251707822084427]]]], [[[[-1.3522599935531616, 0.8126201629638672], [0.16589181125164032, 0.8412489891052246], [-0.30219200253486633, 0.15823400020599365]], [[1.0180106163024902, -0.9497727155685425], [0.39488810300827026, -1.070603609085083], [3.1795475482940674, -0.04148134961724281]]], [[[0.33975324034690857, 1.5945730209350586], [0.005988189484924078, 0.40653282403945923], [0.1888861209154129, 0.9430111646652222]], [[0.25644397735595703, 0.41236668825149536], [0.23610465228557587, -0.5564737915992737], [0.7421198487281799, -1.1596970558166504]]], [[[-1.8563940525054932, 0.42180314660072327], [1.5437475442886353, 1.324867606163025], [0.32083144783973694, -0.946444034576416]], [[-0.03135203197598457, 0.3175634443759918], [1.3918039798736572, 0.23436520993709564], [1.5030736923217773, 0.5003064870834351]]]]], "dtype": "float32"}, {"__nd__": [0.48340409994125366, -0.16209739446640015], "dtype": "float32"}, 1, 0], "e": {"__nd__": [[[[5.76541805267334, -3.758683204650879, 11.032790184020996, -8.051244735717773, 10.976760864257812, 5.916540145874023], [6.405121326446533, 18.592445373535156, 6.258656024932861, -11.627663612365723, 1.6786718368530273, 2.875042200088501]], [[-5.089902400970459, -2.276848077774048, 2.9777684211730957, 4.812046051025391, 3.3668107986450195, 7.8037028312683105], [-3.4501609802246094, 8.497269630432129, 9.844365119934082, 0.3212433457374573, 8.733163833618164, 1.3516480922698975]], [[1.8438665866851807, 3.397322654724121, -5.1819891929626465, -1.3411834239959717, -2.1186506748199463, 10.818690299987793], [8.610074043273926, 2.712181806564331, -1.274165391921997, -8.78844928741455, -2.2023000717163086, -3.9046573638916016]], [[10.428192138671875, 5.803427219390869, -3.540469169616699, 12.035479545593262, 0.009201645851135254, 9.334765434265137], [0.15260541439056396, 2.8800289630889893, -1.3162195682525635, 9.489063262939453, 2.756514072418213, 0.7567852735519409]]], [[[2.1572916507720947, -0.06042158603668213, 1.3618264198303223, 1.3220009803771973, 0.24925124645233154, -11.862936973571777], [-12.57542896270752, -4.706556797027588, 5.707952976226807, -0.8633504509925842, -4.670971393585205, -5.838468074798584]], [[-1.2017827033996582, 9.334614753723145, -8.063448905944824, -2.8727917671203613, -7.587832927703857, 3.368328094482422], [-4.977593898773193, 0.8011895418167114, -2.8018364906311035, 13.418848037719727, -2.272458076477051, 5.338778495788574]], [[3.0131237506866455, -4.7076497077941895, -3.647244691848755, -2.58023738861084, 8.904787063598633, -9.210718154907227], [4.446953296661377, 5.147367477416992, -5.708227634429932, -8.418821334838867, 0.4726584553718567, 0.26391488313674927]], [[-7.045803546905518, 0.9884099364280701, -2.2587239742279053, 2.760018825531006, -3.992872714996338, -6.704043388366699], [2.8932435512542725, -1.8259050846099854, -13.718294143676758, 6.660492420196533, 0.30132341384887695, -5.7473063468933105]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[-0.02731047011911869, -0.6600071787834167, 0.016660654917359352, -0.11380375921726227, 1.3104225397109985, -1.1718063354492188, 0.08312415331602097, -0.050207749009132385, -1.0076088905334473, 1.421728491783142], [0.951816201210022, 0.42787325382232666, -0.6334792375564575, -0.6270087361335754, -0.8134365081787109, -2.0019021034240723, 2.126420497894287, 0.8243647217750549, 0.8862961530685425, -0.7214399576187134], [-0.8047484159469604, 0.6803531646728516, -0.6526366472244263, 0.8597277402877808, -1.0227115154266357, -0.4308694005012512, -0.33726766705513, -0.22670899331569672, -1.3083815574645996, 0.4641980826854706], [-0.02741391956806183, -2.9739866256713867, -1.9940038919448853, 0.06396452337503433, -0.9023783802986145, -0.7528685331344604, -2.0529704093933105, 0.3754860758781433, 0.35248562693595886, -0.14417722821235657], [-0.5336189270019531, -0.3718806803226471, 0.9519839882850647, 0.5178886651992798, 0.5824435949325562, -0.12574021518230438, 0.020585067570209503, -0.2775871455669403, 0.7233420014381409, 0.7289493680000305], [-1.9616258144378662, 0.6800057888031006, 1.9298303127288818, 0.48619529604911804, 0.025621477514505386, 0.2315710037946701, -0.1221044659614563, 0.9174463152885437, 0.9443029165267944, -0.5379931330680847], [-0.5001683235168457, 1.4005813598632812, -1.313825249671936, -0.13662642240524292, 0.8187440037727356, -0.3619335889816284, 0.17222562432289124, 1.5919777154922485, 0.3833082914352417, 1.1751891374588013], [-0.4565652310848236, 0.20578806102275848, -2.322219133377075, 0.6268729567527771, -0.5518076419830322, -0.1923539936542511, 1.0071024894714355, -2.109341859817505, 0.1854449361562729, 0.9098367094993591], [0.95663982629776, -2.0233266353607178, 0.5448036789894104, 1.3278546333312988, -0.9851896166801453, 0.48676422238349915, 0.48642972111701965, -0.23134200274944305, -0.19637636840343475, 1.2644795179367065]], [[0.40314429998397827, -0.180122509598732, -0.9893970489501953, 1.1749540567398071, 2.2566864490509033, 1.4686208963394165, -2.347317934036255, -1.2463041543960571, 0.3099433183670044, 0.14120705425739288], [-0.16195844113826752, -2.247169017791748, -0.9106709361076355, 0.49273496866226196, -0.906436562538147, -1.4499353170394897, -0.3061593472957611, 0.2629493176937103, 0.4885641932487488, -0.9669395089149475], [-1.1329981088638306, -0.16073377430438995, 0.3042134940624237, 1.0176364183425903, -0.9840990304946899, -0.5231059193611145, -1.8025795221328735, -1.6693426370620728, 1.57660710811615, 0.145736962556839], [0.804456889629364, 0.15919823944568634, -0.7718403935432434, 1.0892024040222168, 1.1700375080108643, 0.4181826412677765, 1.4035879373550415, 0.23203034698963165, -0.05100543051958084, -0.034589026123285294], [1.642909049987793, -0.5101089477539062, 1.3900439739227295, 1.2103828191757202, -0.3260663151741028, 2.6671183109283447, -0.10876362770795822, -0.8027604818344116, 1.0832964181900024, -0.24986036121845245], [0.7562732100486755, -0.13955754041671753, -1.553619146347046, 0.759016215801239, -0.5405297875404358, 0.651075541973114, -0.20479990541934967, -0.6427812576293945, 0.03975893557071686, 0.9559571146965027], [0.5527997612953186, -0.24516801536083221, -2.525355100631714, 0.7946208119392395, 0.08896427601575851, 0.8554133176803589, -0.33507707715034485, -0.07507558912038803, -0.3121242821216583, 0.26332008838653564], [-0.08722271025180817, -0.4093080461025238, 1.8730136156082153, -0.7588939070701599, 0.7947620749473572, 0.4831961691379547, -1.281440019607544, 1.9183346033096313, -0.2124776691198349, -0.5433036088943481], [0.6034847497940063, 0.09439679980278015, -0.2875528335571289, -0.26370111107826233, -0.34035196900367737, -2.0223143100738525, 1.5351260900497437, 1.0954921245574951, -0.23191797733306885, -0.9212136268615723]], [[0.34487324953079224, 1.084376335144043, 0.9405297040939331, 0.486558198928833, -0.10963659733533859, 0.20566895604133606, 0.2526969015598297, 0.05455775558948517, 0.5555232763290405, 0.7086456418037415], [1.2119545936584473, 1.0390383005142212, -0.1567617505788803, 0.10046829283237457, -0.4904486835002899, 0.4502304792404175, -0.8104223012924194, 1.679840326309204, -1.1647846698760986, 0.39639660716056824], [0.44976431131362915, 0.5868809223175049, 2.9891903400421143, -0.3453468680381775, -0.39630889892578125, 0.507346510887146, -0.8636246919631958, 0.3899727165699005, 2.788048028945923, 0.08329339325428009], [0.2568548321723938, 0.43716174364089966, -0.05340449512004852, -0.7735477685928345, -1.3057154417037964, 1.8039861917495728, -0.21965591609477997, 0.3362087607383728, -0.9244131445884705, -0.626593828201294], [0.5715702772140503, -0.4028361737728119, 0.9135953783988953, 0.8058263063430786, -0.8755656480789185, 1.4999866485595703, -0.11816059798002243, 0.9673881530761719, -1.4313210248947144, -0.12003026157617569], [0.9585743546485901, -0.026693129912018776, -0.04468448460102081, -0.06471893936395645, 2.1490702629089355, -0.3245409429073334, 0.6320807933807373, -0.5381097793579102, -0.383578360080719, 0.5740225315093994], [1.2868865728378296, -1.5695816278457642, 0.5107666850090027, 0.19681452214717865, 0.026298601180315018, -0.021379247307777405, -0.9622975587844849, 0.9332522749900818, -0.6319559216499329, -0.3549959063529968], [-0.48573288321495056, -1.2120885848999023, -0.3851938545703888, -1.376354455947876, -0.9344764351844788, 0.40188437700271606, -0.32389092445373535, -0.20764216780662537, -0.4909537136554718, 0.5215466022491455], [0.7770124673843384, 0.02942526713013649, 0.6949523687362671, -1.836419939994812, 0.5129597783088684, 0.17022588849067688, -0.5007901191711426, 1.0383484363555908, 0.8021070957183838, -0.36663246154785156]], [[-0.2617701292037964, 0.36271145939826965, -0.37193745374679565, -0.5687625408172607, -1.6746898889541626, 0.8730651140213013, -0.7516005635261536, -0.5054689645767212, 0.37985438108444214, -0.24499565362930298], [0.9657680988311768, -0.0979146808385849, -1.8673925399780273, 0.9601231217384338, 0.25423818826675415, -0.01546180434525013, 0.2834683358669281, -0.2186422348022461, 0.859622597694397, 0.661320686340332], [0.40580013394355774, 0.49477219581604004, -1.523177146911621, 0.4315924048423767, -2.383768081665039, 0.3844493627548218, 1.3804408311843872, 2.0462160110473633, -0.06923765689134598, -0.2055208683013916], [-1.342581868171692, 1.624799132347107, -1.6947627067565918, -0.5952433943748474, -0.32226794958114624, -0.7775636911392212, 0.36694324016571045, 0.3450537919998169, 0.135455921292305, 1.3521013259887695], [0.6694877743721008, -0.49809175729751587, -0.7169726490974426, 2.1718616485595703, -1.3404685258865356, -0.3798496723175049, 0.0854981392621994, -1.4832100868225098, 0.5055801868438721, -0.8671334385871887], [1.971063494682312, -0.3715318441390991, 0.7691224813461304, 0.3326915204524994, -0.08465104550123215, -0.43265092372894287, -0.4707046151161194, 1.0865046977996826, 0.7793607711791992, -0.25108757615089417], [0.23296865820884705, -0.7603147625923157, -0.1709664911031723, -0.6086498498916626, -0.47891849279403687, 2.356762409210205, 0.23733220994472504, -1.5883610248565674, 2.107896566390991, -0.44715678691864014], [0.9650080800056458, -0.6561185121536255, 0.039395641535520554, -0.6036797165870667, 0.061589550226926804, 1.4278604984283447, 1.6722804307937622, 2.0452699661254883, -0.3058551549911499, 0.47801679372787476], [-0.32433322072029114, -0.15547962486743927, 1.8352606296539307, 0.5427495241165161, -2.321866750717163, -0.12672452628612518, 0.5074414014816284, 0.7670865654945374, -0.47581201791763306, -0.9271156191825867]], [[-1.7602956295013428, 0.8293377161026001, 0.15931633114814758, -1.2283183336257935, 0.7925553321838379, 1.50795578956604, -0.6152002811431885, 1.2835344076156616, -1.1445600986480713, 0.45009085536003113], [-0.19814561307430267, 1.0277135372161865, 0.13675495982170105, 1.5419833660125732, 0.03329082950949669, -1.2631491422653198, 0.344074010848999, 0.09253962337970734, 1.0167542695999146, -0.9760038256645203], [0.9548048377037048, -1.0269368886947632, -2.0112075805664062, -0.45075663924217224, 0.05553658306598663, 1.8923943042755127, 0.11567860841751099, -0.797221839427948, 1.4080222845077515, 1.518416404724121], [1.249260663986206, -1.4187119007110596, -0.4773699939250946, 0.9541443586349487, -1.6672111749649048, 0.055854447185993195, 0.04199981689453125, 0.013737411238253117, 0.05825261399149895, 0.3736379146575928], [-0.42374491691589355, 1.378548502922058, 0.14193366467952728, 0.7754107117652893, 1.3100290298461914, 1.392555594444275, 0.7724257111549377, 1.7456796169281006, -0.17029798030853271, 0.2378583401441574], [-0.07885283976793289, -0.3677898943424225, 0.440060555934906, -0.18821890652179718, 0.15602940320968628, -0.2689659297466278, -1.178993582725525, 0.831200897693634, 1.0348224639892578, 0.3682132661342621], [0.18349507451057434, 0.40185245871543884, 0.1452648788690567, 1.0599623918533325, -0.5450511574745178, -0.11202820390462875, -0.7347216606140137, -1.2554198503494263, -0.6720921397209167, 0.3375609815120697], [-0.45568767189979553, -1.4453189373016357, -0.1094651147723198, 0.7909508943557739, 1.6267569065093994, 0.9893428683280945, -0.8020105361938477, 1.0344520807266235, -2.1381866931915283, 0.2458416372537613], [0.7082198262214661, 1.1504642963409424, 0.22312186658382416, -0.797712504863739, 1.1496119499206543, 1.2565094232559204, 0.27656763792037964, -1.5847113132476807, -0.6740804314613342, -0.5137516856193542]], [[-1.3084962368011475, -1.181649088859558, -1.038490891456604, -0.11008056998252869, 0.22820107638835907, 0.9073446393013, 0.27398207783699036, 1.5481576919555664, 1.4335856437683105, -0.2821529507637024], [0.48400819301605225, 0.46994835138320923, -0.034830573946237564, 0.2641181945800781, 0.5649716854095459, -0.29572510719299316, 0.03963874652981758, 0.4375014901161194, -0.49384790658950806, 1.44437575340271], [-1.3460843563079834, 1.084398865699768, -0.2669949531555176, -1.9248790740966797, -0.3103373944759369, 0.0360223613679409, -1.3134183883666992, -0.2372247725725174, 0.14984889328479767, -0.8602626919746399], [0.6166486144065857, 0.6099576950073242, -0.633034884929657, -0.4745877981185913, 0.767430305480957, 0.07653787732124329, 0.5999372005462646, -0.5921158790588379, -0.6627402901649475, 0.7002205848693848], [0.1972624510526657, 0.5025166273117065, 1.685814380645752, 0.4048061966896057, -0.1303189992904663, -1.6746212244033813, 0.27698540687561035, 0.5200858116149902, 0.22143813967704773, 1.1771094799041748], [1.5865280628204346, 0.21322549879550934, -0.6809890270233154, -0.45849207043647766, 0.020999249070882797, 1.3756108283996582, 0.16729792952537537, 1.320385217666626, -1.1234748363494873, 1.2470191717147827], [0.6521049737930298, -0.23082496225833893, 0.4470270574092865, 1.1180700063705444, 0.6743082404136658, 1.1417038440704346, 0.6131512522697449, 0.8310059905052185, 1.0453847646713257, 0.1318836659193039], [0.1500466763973236, 0.7562556862831116, 0.7700712084770203, -1.0577754974365234, -0.6119639277458191, -1.1457581520080566, -0.22200323641300201, -0.8088290095329285, 1.37744140625, -1.4322701692581177], [-0.27533024549484253, 0.3538517355918884, -1.7304182052612305, 0.9343510866165161, -0.22732006013393402, -0.037871427834033966, 0.5701348781585693, 2.2180540561676025, -0.11502029746770859, -0.47705045342445374]], [[0.6452286243438721, -1.2337216138839722, 0.5074419975280762, 0.5249574184417725, -1.678446888923645, -0.4883384108543396, 1.575673222541809, 0.7507332563400269, -0.03053421340882778, 1.9606841802597046], [-2.074122667312622, -0.06664688140153885, 0.6264083385467529, 0.44504743814468384, 0.5134199857711792, -0.7018601894378662, -0.8498713970184326, 1.9003005027770996, 0.40764179825782776, 0.8183737993240356], [-0.7729602456092834, -0.8557814955711365, -0.3554303050041199, 0.9730470180511475, 0.5771521329879761, -0.5115997791290283, -0.5322303771972656, 1.5490999221801758, 0.19931501150131226, 0.4498286843299866], [0.9734077453613281, 1.5507402420043945, -0.22585132718086243, 0.7700297832489014, -0.43982335925102234, 2.0987257957458496, 0.43209967017173767, 0.09472857415676117, 0.2846527695655823, -0.6286530494689941], [0.6450279951095581, 0.0830150917172432, 0.38494327664375305, 0.5659579634666443, -0.2599296271800995, 0.19029048085212708, 1.5583231449127197, 1.9515631198883057, -0.25146129727363586, 0.5221169590950012], [-0.312272846698761, 0.3665103018283844, 2.0818166732788086, -1.7254995107650757, -0.871235191822052, 0.9971799850463867, 1.6909842491149902, 0.32263240218162537, -1.3353466987609863, -0.7073681354522705], [-0.3008096218109131, 1.571364402770996, 1.1946922540664673, 0.6872475743293762, 3.1719555854797363, -1.646079659461975, 0.27750515937805176, -0.11646474897861481, 1.074398398399353, 0.4323534667491913], [-0.45388782024383545, 1.2770843505859375, 0.12034802883863449, -0.4221932291984558, -0.19321979582309723, -1.2799134254455566, 1.0438913106918335, -1.6885536909103394, -0.07683508098125458, 0.8865239024162292], [0.9025394320487976, 0.8326643705368042, 0.40137141942977905, 0.32493162155151367, -1.249244213104248, -0.6594018936157227, -1.4879403114318848, 1.2527153491973877, 0.21113267540931702, 1.5097121000289917]]], [[[1.3112448453903198, -0.12459177523851395, -0.29125261306762695, 0.15481680631637573, -0.39323464035987854, 0.45520350337028503, 0.9310211539268494, -0.26385298371315, 1.7679697275161743, -0.762860119342804], [-1.1044753789901733, -0.48534300923347473, 0.634936511516571, 0.3864682912826538, -1.0909335613250732, -0.6068747639656067, 1.2402552366256714, 0.5230608582496643, -2.166438102722168, 0.27121421694755554], [-0.17498956620693207, 0.14076340198516846, 0.9339569807052612, 1.1140490770339966, 0.4587056338787079, 1.213120937347412, -0.7705376148223877, -0.2677188217639923, 1.113088846206665, -1.673121690750122], [0.14334037899971008, -0.5178122520446777, 0.15837620198726654, -0.03127862885594368, -0.7076389193534851, 0.6308673024177551, -1.2259873151779175, 0.4234137237071991, -0.06235792487859726, -1.3647334575653076], [0.6492087244987488, -0.3041384816169739, 1.0569771528244019, -0.14371201395988464, 0.32022103667259216, 2.201437473297119, -0.0011069821193814278, 0.5571980476379395, 0.7061403393745422, 1.2240092754364014], [-0.09437218308448792, -0.09104501456022263, -1.2604745626449585, -0.3134593963623047, -0.4796338379383087, -0.5970747470855713, 0.03774450346827507, 0.6265586018562317, -1.0146541595458984, 1.6636074781417847], [1.305154800415039, 0.3128124475479126, 0.9667807817459106, -1.4836061000823975, -0.301615446805954, -0.07992472499608994, 0.1706649214029312, -0.6357991695404053, 0.07454049587249756, -0.008642049506306648], [1.68630850315094, 0.7228384613990784, -0.04646238312125206, -0.18475459516048431, -0.4743667542934418, 0.07757454365491867, -0.8210821151733398, 0.23187072575092316, 1.7113744020462036, -0.9180041551589966], [-2.014450788497925, 1.3465343713760376, -1.1249604225158691, -1.6723394393920898, 0.11680914461612701, -1.0844916105270386, -0.7879948019981384, 0.52069491147995, 0.5671910047531128, 0.38080349564552307]], [[-0.21461006999015808, 0.8695266842842102, 0.43844926357269287, -0.26774147152900696, 1.5032988786697388, -0.2776227593421936, -0.28794264793395996, -0.3020627498626709, 0.15825685858726501, -0.2251136153936386], [0.030152857303619385, -2.6855227947235107, 1.2132396697998047, 0.30714190006256104, 0.744663417339325, -0.7386519312858582, 2.282026767730713, 1.8621960878372192, -1.221285104751587, -0.8150803446769714], [-1.6550811529159546, 1.033289909362793, 0.7074235677719116, -0.18007950484752655, 2.45365571975708, -2.3328590393066406, -2.0196080207824707, 1.0254658460617065, 1.4862719774246216, 0.7180656790733337], [1.0522032976150513, 1.0526334047317505, -2.0616137981414795, -0.9282088279724121, -0.6845653653144836, -0.807471752166748, -1.1412076950073242, 0.8644338250160217, 0.17701151967048645, -0.3213983178138733], [-1.313594937324524, 0.05538565665483475, 0.5521374344825745, 0.7033114433288574, 1.0724271535873413, 0.5459430813789368, -0.3302827477455139, 0.43206191062927246, -1.4545888900756836, -0.3511306941509247], [1.4028784036636353, 0.29296550154685974, 1.5197256803512573, -0.007910938002169132, -0.6993071436882019, 2.9608891010284424, -0.6653543710708618, 0.15363679826259613, 2.367499589920044, -0.7124156355857849], [-1.564648151397705, -0.3651380240917206, 0.8683984279632568, -1.0779801607131958, -1.4984405040740967, -0.17910946905612946, -0.6429880261421204, 0.18525172770023346, 0.3602816164493561, -1.5843188762664795], [1.3146674633026123, -0.5921146869659424, -0.6175894141197205, 0.7943690419197083, 0.12965300679206848, 0.4561786949634552, 0.781883716583252, 0.49988359212875366, 0.24383606016635895, -1.2283368110656738], [-0.8607149720191956, -0.5142956376075745, 0.4772038757801056, 0.6820409297943115, 0.4670776128768921, 0.5385591387748718, -0.9665507078170776, -0.48943397402763367, -1.3014566898345947, -0.2621902823448181]], [[-1.454444408416748, -0.46239960193634033, -2.1152312755584717, -0.6192602515220642, -0.41031602025032043, 1.1647032499313354, -0.5226447582244873, 0.48319175839424133, 1.4403891563415527, 1.4172652959823608], [0.4862161874771118, -1.6357319355010986, -0.43220242857933044, 0.8642532825469971, 0.03810169920325279, 0.6876357793807983, -1.0522730350494385, -0.370033860206604, 0.41742342710494995, -1.4410334825515747], [0.2943457067012787, 0.20873069763183594, -0.9846866726875305, 0.7974342703819275, 1.374219298362732, 1.052905559539795, 0.10760953277349472, -0.05393601581454277, 0.3310606777667999, 1.2232416868209839], [-0.8754714727401733, 1.406256914138794, -0.21156705915927887, -0.23777155578136444, -0.3528509736061096, 2.098296880722046, -0.724128782749176, 0.9670973420143127, 0.18824365735054016, -0.8238368034362793], [-0.5542449951171875, 1.2405563592910767, 0.5639222264289856, -0.1198815330862999, -0.19221188127994537, -0.44248706102371216, 1.1620782613754272, 1.2263851165771484, -1.5188884735107422, 0.2739797532558441], [0.4407263994216919, -1.1116431951522827, -0.572824239730835, -0.23746643960475922, 0.40962475538253784, -0.21150514483451843, -0.3001190721988678, -1.1000005006790161, 0.6853576302528381, -0.038948915898799896], [-0.13824652135372162, -0.2924332022666931, -1.0820438861846924, -1.28849196434021, 0.6609392166137695, -0.9237057566642761, -0.6787837147712708, -1.3648922443389893, 0.6923794746398926, 0.7586212754249573], [0.8986443877220154, -0.598563015460968, 0.175231471657753, 1.145768165588379, -0.9840343594551086, -0.9497076272964478, 1.1901081800460815, -0.8419787883758545, -2.332427740097046, -1.5351898670196533], [-1.3772436380386353, -0.7532730102539062, -0.3289835751056671, 0.30259934067726135, -0.920737624168396, 0.13436675071716309, 1.06133234500885, 0.3807411789894104, 3.56795072555542, 1.3336410522460938]], [[-0.0629827231168747, 0.18392322957515717, -0.5850589275360107, 0.05539698153734207, 0.12413384020328522, -0.16939441859722137, 0.46009400486946106, 1.1972423791885376, 0.09042444080114365, -0.9146829843521118], [0.6871934533119202, 0.572272777557373, 2.2127630710601807, -0.14643393456935883, -0.6186737418174744, 0.5244547724723816, -0.30424413084983826, 0.4304141700267792, 1.1028499603271484, 0.8142206072807312], [1.2329070568084717, 1.0589667558670044, -0.7123379707336426, 0.16807222366333008, -0.33598390221595764, 0.21161434054374695, -0.9054358005523682, 0.0611727274954319, -0.19992004334926605, 0.6704902052879333], [-1.1615660190582275, -1.383392572402954, -1.0069384574890137, 0.009429015219211578, -0.5601580739021301, -0.7176916003227234, -0.8949154019355774, 1.691205620765686, -1.1715891361236572, -0.4636026620864868], [1.3886442184448242, -0.24050793051719666, -0.8866854310035706, -0.7773311734199524, -0.9875670075416565, 1.4897619485855103, 1.4184918403625488, 0.3282889723777771, 1.7440794706344604, -0.41498491168022156], [0.3410469889640808, 0.6407409906387329, 0.3219364881515503, -0.012612254358828068, -0.9388310313224792, 0.38674744963645935, -0.17498895525932312, 0.5438105463981628, -2.2721762657165527, 0.36195576190948486], [-0.2933202087879181, 0.34699365496635437, -1.0106279850006104, 0.883223295211792, 1.1741821765899658, -1.0216264724731445, -2.3130147457122803, -1.5588414669036865, -0.5646703243255615, -0.03794748708605766], [0.9644319415092468, 1.3001668453216553, 0.17052499949932098, -1.0348589420318604, 0.3556004762649536, -0.834872305393219, -1.646485447883606, 0.9472529888153076, 0.23379051685333252, -0.23919646441936493], [0.9747881889343262, -0.4694947898387909, 0.4373168647289276, -0.5223269462585449, -0.8887767791748047, 0.951626718044281, -0.7254688739776611, -1.3334901332855225, 0.9523716568946838, 0.5402659773826599]], [[1.0461560487747192, -1.9050681591033936, 0.29636624455451965, -1.2944202423095703, -0.40244773030281067, -0.26997101306915283, -0.8096369504928589, -0.8026348352432251, -1.338985800743103, -0.2305590659379959], [-1.9215677976608276, -2.229032039642334, 0.5666467547416687, 0.9450455904006958, 0.0996093675494194, -0.5653381943702698, -1.0297372341156006, -0.6826286911964417, -1.9779504537582397, 0.09357182681560516], [0.9249580502510071, 1.8011976480484009, 0.3225092887878418, -1.136582851409912, 1.6316698789596558, 1.2258753776550293, 0.5363245010375977, -0.9660948514938354, 0.2496425360441208, -1.1571065187454224], [0.15935957431793213, 0.9544628262519836, -0.4429158866405487, 0.6198605895042419, 0.33259448409080505, -0.13926397264003754, -1.1828991174697876, -1.0023802518844604, -0.6371804475784302, -2.086562395095825], [0.01314813457429409, -0.6543765068054199, 0.2513366937637329, 0.35078948736190796, -0.025717519223690033, 0.786629855632782, -0.9748731851577759, -1.4094657897949219, 0.22761139273643494, 0.2770756185054779], [0.6311420798301697, 0.3472185730934143, 0.32268133759498596, -0.8291063904762268, -1.2927136421203613, 0.16287024319171906, -1.998913288116455, -0.03388554975390434, -0.708509087562561, -2.1368215084075928], [-1.3251891136169434, -1.709580898284912, 2.0888783931732178, -1.4796557426452637, 0.3249761760234833, 0.40084972977638245, -0.5103784799575806, -0.3923795223236084, 0.37624579668045044, 1.916684865951538], [1.0203217267990112, -0.6310632228851318, -1.4188337326049805, 0.6383673548698425, 0.6994829773902893, -1.2520596981048584, -1.7508773803710938, 0.23808439075946808, -1.6417045593261719, 0.06369362026453018], [-2.8638784885406494, -0.5121569037437439, -1.5759854316711426, 0.9911691546440125, 1.61319899559021, 0.16826224327087402, 1.0728284120559692, 0.06641809642314911, 0.5722687244415283, 1.806087851524353]], [[-0.5502312779426575, -1.8213324546813965, -0.005148874595761299, 2.283489942550659, 1.2661100625991821, 2.0418550968170166, 0.06255951523780823, -0.31626468896865845, -1.307456135749817, -0.5933854579925537], [-0.4379447102546692, -0.39421552419662476, 0.8066501617431641, 0.4053013026714325, 1.3750487565994263, -0.502264678478241, -0.5367162227630615, -0.7482833862304688, 1.0612506866455078, -0.1956312358379364], [0.4204866290092468, 0.13539977371692657, -0.4798395335674286, 0.5814936757087708, 1.3475449085235596, -0.4896393120288849, -1.8439393043518066, 0.9699366688728333, 0.8664035797119141, -0.01889997348189354], [0.9574966430664062, 1.2873728275299072, 1.3165193796157837, 1.5543967485427856, -0.31253302097320557, 0.8773953914642334, -0.8592864274978638, 0.06479943543672562, 1.1706972122192383, 0.2095934897661209], [-0.38123857975006104, 0.8329847455024719, 0.8523194193840027, 1.0877456665039062, -2.706838607788086, -1.1539233922958374, 0.3441007435321808, 1.1028186082839966, 0.6258363127708435, -1.1368134021759033], [-0.28303438425064087, -0.2807702124118805, -1.7007853984832764, -1.2596533298492432, -0.8495974540710449, -1.2257511615753174, 1.6882396936416626, -0.6997112035751343, 0.1457880735397339, -0.8574202656745911], [0.2530701458454132, -1.2436518669128418, -1.2743386030197144, 1.643984079360962, -0.19217921793460846, -1.1682243347167969, -1.395063042640686, -0.7591588497161865, 0.964364230632782, 0.8291627168655396], [0.9310388565063477, -0.2982458174228668, 0.3286254107952118, 0.20639188587665558, -0.6174749732017517, -0.5954110026359558, 0.2206779420375824, 0.9688808917999268, 1.4513859748840332, -0.9950783848762512], [-0.2037268877029419, 0.12189039587974548, 1.601920485496521, -1.1854873895645142, -0.3344455063343048, -1.6301119327545166, -0.934933602809906, -1.1034038066864014, -1.6203292608261108, -0.9925321936607361]], [[0.5689323544502258, 0.13575169444084167, 2.1829516887664795, -0.15210992097854614, 1.03878915309906, -2.2973177433013916, -1.9982426166534424, 1.1311006546020508, 0.9827473759651184, -0.30308133363723755], [-0.8849696516990662, 1.1592388153076172, -0.44191035628318787, -0.3660805821418762, -0.9068123698234558, -0.012338340282440186, 0.7442436814308167, 1.2778009176254272, -0.5288992524147034, 0.1965782195329666], [-0.5172207355499268, 1.994565725326538, -1.4643940925598145, -1.6899020671844482, 1.0244262218475342, 0.7385838627815247, 0.6801588535308838, -0.9018722176551819, -0.4713539481163025, 0.501465916633606], [-0.45429328083992004, -0.6258816719055176, -0.5558122396469116, 1.8632776737213135, -2.372843027114868, 0.21843993663787842, -0.7540668249130249, -0.790404200553894, 0.30264294147491455, 0.86191326379776], [-0.4605056941509247, 2.746856212615967, -2.035175323486328, 0.2844705581665039, 0.5304864645004272, -0.5944528579711914, -0.23502089083194733, -0.5209451913833618, -0.230324849486351, 0.020290235057473183], [0.15869081020355225, -2.5676369667053223, -0.033588092774152756, -0.16466861963272095, -2.6831371784210205, 0.7808482050895691, -0.03698981925845146, 0.8600031733512878, -1.082044243812561, 0.9882801175117493], [0.23126034438610077, 2.1827478408813477, -3.2007222175598145, -1.1996794939041138, -2.9606127738952637, 1.5272208452224731, 0.3435983955860138, 2.1198196411132812, 0.8987952470779419, -0.6163374185562134], [1.6662238836288452, -0.39394882321357727, 0.145126074552536, 2.468116521835327, -0.7738761305809021, 1.0213109254837036, -0.24924099445343018, 0.7684609293937683, 0.19167469441890717, -0.1307147592306137], [1.2904739379882812, 1.498754620552063, -2.0168650150299072, -1.8344247341156006, -0.9334522485733032, 0.7591123580932617, -0.13658258318901062, -1.5346102714538574, 0.34093788266181946, -0.5404829978942871]]]], "dtype": "float32"}, {"__nd__": [[[[[-0.9504693150520325, -1.224109411239624, -0.2843164801597595], [-0.380435585975647, 1.1766571998596191, -1.2726694345474243], [-0.13349714875221252, 0.22774624824523926, -0.6215961575508118]], [[0.4295446574687958, 1.2298095226287842, 0.13670776784420013], [-0.2004203051328659, -1.2978317737579346, -0.6769388914108276], [-0.11681350320577621, 0.9851891398429871, 0.09830936044454575]], [[-0.62091463804245, 0.13633063435554504, 0.5572341680526733], [-0.09827329963445663, -0.4546144902706146, 0.2777976989746094], [0.26668527722358704, -2.0613112449645996, -0.9946417808532715]]], [[[-1.6116691827774048, 1.7759344577789307, 0.008884531445801258], [-0.3736242949962616, -0.462209552526474, -0.15494653582572937], [0.12185267359018326, -1.1171449422836304, 0.6538978219032288]], [[-0.8923137187957764, -1.434787392616272, 2.599130630493164], [-1.272359013557434, 0.4020821452140808, 0.18039163947105408], [0.1798269897699356, -0.6347090601921082, -0.22542616724967957]], [[0.9579179883003235, -0.537982702255249, -0.2019181251525879], [0.23037412762641907, -0.07082907855510712, -1.2485196590423584], [1.4367648363113403, 1.1129387617111206, 0.5287482142448425]]]], [[[[-0.14610058069229126, -0.4545418918132782, 0.014711618423461914], [-1.450918436050415, -0.9413972496986389, -0.3835086226463318], [-0.1110559031367302, -0.16488954424858093, -0.6541319489479065]], [[0.4968682825565338, -0.4607379138469696, -0.3831747770309448], [2.3226373195648193, -1.3282487392425537, 0.044105466455221176], [2.435445547103882, -0.7553452849388123, -1.655365228652954]], [[-1.437441110610962, 0.38248005509376526, -0.27495166659355164], [1.2271735668182373, 0.405990868806839, 0.7370147109031677], [-0.2732369899749756, -0.1198788657784462, 0.34457799792289734]]], [[[0.4149656593799591, -0.9803626537322998, -0.8078548908233643], [2.009554862976074, 0.16533935070037842, -2.323991060256958], [-0.7841596603393555, 0.3969343602657318, -0.6608436107635498]], [[1.2091110944747925, -1.9079455137252808, 0.226007878780365], [-0.704684853553772, 1.3896024227142334, 1.0786268711090088], [0.6440380811691284, 0.36628103256225586, 1.6437699794769287]], [[-2.0882275104522705, 0.15291716158390045, 0.5144861936569214], [1.8219090700149536, 0.36002084612846375, 1.0763949155807495], [0.6395936012268066, 0.4389919340610504, -0.58163982629776]]]], [[[[0.8902316689491272, -1.585006833076477, 0.10813190788030624], [-0.3133467435836792, -2.1370630264282227, 0.056707996875047684], [1.1718637943267822, 0.7322306632995605, 0.5492442846298218]], [[-0.02430884726345539, -0.7173755168914795, -0.5659865140914917], [-2.002044200897217, 1.4712169170379639, 0.804969072341919], [-0.4911953806877136, 0.10170886665582657, -1.5503100156784058]], [[1.0726628303527832, -0.6829448342323303, -0.3320455849170685], [-2.253458023071289, -1.6758456230163574, 0.1499268263578415], [-0.4949462413787842, 0.3911455571651459, 0.5201542377471924]]], [[[-0.2042209953069687, 0.09780041873455048, 1.343245506286621], [1.9812061786651611, 0.16905754804611206, 0.6206749677658081], [-0.21901936829090118, -1.7497279644012451, 0.47258836030960083]], [[-1.066673755645752, -1.3328694105148315, -0.792037308216095], [0.29211440682411194, 1.020511507987976, -0.08469045162200928], [-1.03555428981781, -0.8381768465042114, 0.21179993450641632]], [[-0.1882709413766861, -0.2021469622850418, -0.21977613866329193], [-0.8156774640083313, -0.6791977882385254, -0.9771034717559814], [0.1900426149368286, -0.6608357429504395, 1.0433712005615234]]]], [[[[1.7966867685317993, -0.13941611349582672, 0.3399154543876648], [0.5650801062583923, -0.6933538317680359, -0.9059010148048401], [0.8732523322105408, 0.6371913552284241, 1.00072181224823]], [[-0.18091754615306854, 0.3825458586215973, -0.6263382434844971], [-1.2482578754425049, 0.40811336040496826, -1.526369571685791], [-0.31966632604599, 1.0067741870880127, -0.657130777835846]], [[1.6862503290176392, -1.643067717552185, -0.26368463039398193], [1.0236570835113525, 0.7911830544471741, 0.24018742144107819], [-0.026125626638531685, -0.22288423776626587, 0.023537233471870422]]], [[[-1.6826965808868408, 0.01905490644276142, -1.517408013343811], [0.3547455966472626, -0.2940763533115387, -1.7362257242202759], [2.5194509029388428, 0.25066617131233215, -0.4903227984905243]], [[0.5333260297775269, 0.5514765977859497, 0.3260079622268677], [1.1533957719802856, 0.17204663157463074, -0.8401803970336914], [1.2029796838760376, -2.112583875656128, 0.028101423755288124]], [[-0.213910311460495, -1.1777863502502441, -0.6053476333618164], [-0.16433387994766235, 1.8149083852767944, 0.5387120842933655], [-0.5649818181991577, -1.274820327758789, -1.2932345867156982]]]]], "dtype": "float32"}, {"__nd__": [0.6618387699127197, 1.213690161705017, -0.4877850115299225, -0.0036227311939001083], "dtype": "float32"}, 1, 1], "e": {"__nd__": [[[[3.3178675174713135, 2.2556705474853516, -0.6548092365264893, 0.3269317150115967, 3.2445406913757324, 4.646275520324707, 5.322079658508301, 5.687697410583496, 5.171297073364258, -5.494245529174805], [-0.47603869438171387, -3.779879331588745, 3.422571897506714, 2.855956554412842, 8.002766609191895, 3.396449089050293, -7.36112117767334, 3.951824426651001, -7.348199844360352, 10.313013076782227], [0.9197359085083008, 3.900200605392456, -2.655200481414795, -15.635696411132812, -4.8302412033081055, 2.375706672668457, -7.233834266662598, -2.0153005123138428, 10.14476203918457, -1.2000130414962769], [-4.09501838684082, 7.675591468811035, 6.29534912109375, 1.6854403018951416, 1.6037647724151611, -1.0999534130096436, -0.6476335525512695, 2.6771743297576904, -9.173553466796875, 0.19476479291915894], [-3.1136245727539062, 2.9385621547698975, 5.168967247009277, -6.173539161682129, 3.4455840587615967, -3.1971213817596436, 1.0482758283615112, 3.303779363632202, -0.9673655033111572, -0.01128929853439331], [-4.032576560974121, -0.8853628635406494, 0.930099606513977, 0.31275713443756104, 0.7607421875, -5.642909049987793, -0.14967751502990723, -0.9574451446533203, 4.432954788208008, -0.5419355630874634], [-1.9437413215637207, -7.125179290771484, 1.6819931268692017, 7.241569519042969, -0.07906705141067505, 2.678966760635376, 7.806210517883301, -11.423803329467773, 10.27289867401123, 1.898555874824524], [1.0175011157989502, -3.327651262283325, -3.5021040439605713, 8.3872709274292, 6.364343643188477, 3.457347869873047, -5.872678756713867, 0.6834980249404907, 3.0541343688964844, 1.0956190824508667], [-1.2239587306976318, 4.5106353759765625, -8.010551452636719, -3.2888059616088867, 4.453796863555908, -1.7518539428710938, 4.944027900695801, 7.116926193237305, -6.7391557693481445, 0.592796802520752]], [[-1.4053966999053955, 0.41057583689689636, -2.911292552947998, -4.234066963195801, 1.9252040386199951, -5.478240013122559, 0.8258392810821533, -5.2165937423706055, 1.9001591205596924, -1.9657995700836182], [10.236374855041504, -0.8583981990814209, -5.065406799316406, 5.729937553405762, 3.9650399684906006, 3.5922718048095703, 10.106616973876953, -13.8041353225708, -1.0842063426971436, 3.0112056732177734], [-10.232266426086426, 12.221414566040039, -3.5104949474334717, 3.713451623916626, 0.2696704864501953, 3.2143521308898926, 18.872148513793945, -6.323083877563477, -12.672523498535156, 8.923317909240723], [7.352320671081543, 2.6539645195007324, -3.1661312580108643, 7.567144393920898, -12.153371810913086, -3.7353293895721436, 1.135287880897522, 7.945624351501465, 11.042525291442871, -10.095415115356445], [-3.195392608642578, -9.096930503845215, 1.6287782192230225, -0.7278016805648804, -2.2620491981506348, -2.8561885356903076, 3.9255077838897705, 1.8962395191192627, -3.9540441036224365, 0.16673174500465393], [1.9370148181915283, 4.030755519866943, 1.2143807411193848, -1.301386833190918, -0.3589131832122803, 8.728349685668945, -9.565797805786133, -5.038212776184082, 4.613375663757324, -0.6408653259277344], [-2.0309534072875977, 17.501022338867188, -4.100583076477051, -2.578922986984253, 13.345254898071289, -8.441781997680664, 2.588369369506836, 9.198214530944824, -20.279190063476562, 1.7404643297195435], [6.572836875915527, -5.1549072265625, -11.474931716918945, 3.7456815242767334, -1.612647533416748, -1.8320491313934326, 13.158641815185547, -2.68222975730896, -4.030691146850586, 8.165067672729492], [3.3008079528808594, -3.032404899597168, 6.560605049133301, 7.137456893920898, 3.727766513824463, 4.031534194946289, -6.6353349685668945, 1.493175745010376, -1.4602396488189697, -0.48112237453460693]], [[-0.4248462915420532, 11.870203971862793, -0.666715145111084, 0.9167447686195374, 0.46603551506996155, 5.604800224304199, -5.251720428466797, -0.06517642736434937, 1.8366549015045166, 1.9701604843139648], [4.169817924499512, -2.5526058673858643, 12.985098838806152, 4.3064680099487305, 2.1418254375457764, -8.7921142578125, -3.9604055881500244, -0.3560296297073364, 10.250832557678223, -5.325179100036621], [-5.74227237701416, -11.568685531616211, 12.212834358215332, 4.647561073303223, 0.8092927932739258, -1.8464913368225098, 8.349761962890625, -5.234543800354004, -14.472570419311523, -2.6693942546844482], [4.9341583251953125, 6.382257461547852, 2.976829767227173, 0.36721718311309814, 7.429045677185059, -2.708587169647217, 4.994948387145996, 15.463342666625977, 10.555327415466309, -0.21751165390014648], [7.716036796569824, 3.104832649230957, -12.340957641601562, -1.1198222637176514, 0.1317119002342224, -8.102340698242188, 2.7018632888793945, -6.9721221923828125, -6.884069442749023, 2.1497886180877686], [0.7346940040588379, 3.7860524654388428, -2.255969285964966, 3.815744161605835, -7.287134170532227, -4.938048362731934, -4.117528915405273, -2.1010050773620605, -13.963326454162598, 3.7203807830810547], [-2.2613565921783447, 7.052698135375977, 4.155056953430176, -0.9763426780700684, 1.9752447605133057, 4.566385269165039, -14.697898864746094, 5.849501609802246, 8.781115531921387, -6.188962936401367], [1.2453268766403198, -2.662118673324585, 8.628233909606934, 0.7140234708786011, 5.464049339294434, 3.484419584274292, -11.809507369995117, 3.095080614089966, 4.60725212097168, -2.439530611038208], [-1.4382469654083252, -2.3961169719696045, 3.2436530590057373, -1.6634552478790283, -4.0756072998046875, 5.895593643188477, 2.627445936203003, -12.56721305847168, -1.5279757976531982, 0.8492679595947266]], [[-1.6683013439178467, -5.181689262390137, -3.024453639984131, 6.235174179077148, 5.4795942306518555, -0.04150867462158203, 0.5085889101028442, -2.5284335613250732, -8.125151634216309, 1.1502314805984497], [0.31139588356018066, 8.4635009765625, 1.5122156143188477, -1.5498387813568115, -7.831809997558594, 8.915741920471191, 0.7303768396377563, 6.681502342224121, -12.974862098693848, -6.655241012573242], [0.9309899806976318, -0.7166070938110352, -0.2847630977630615, -0.2225264310836792, 6.568325042724609, -2.467548370361328, -5.2161149978637695, -4.815774917602539, 3.5258076190948486, -8.832353591918945], [-0.8253365755081177, -11.720653533935547, -2.3391144275665283, 5.576517105102539, -7.9055891036987305, 0.8015679717063904, -3.6356284618377686, -1.0576626062393188, -0.891014575958252, -2.614438533782959], [-0.9773058891296387, 1.387347936630249, 0.5056707859039307, -2.1008849143981934, -3.9535744190216064, 7.206043243408203, 0.22432339191436768, -3.1325056552886963, -5.545222282409668, -0.6236892938613892], [-5.0534467697143555, -5.470907211303711, -5.336475372314453, -1.5685162544250488, 6.715195655822754, 8.849662780761719, 6.278909683227539, 3.6108694076538086, -7.47335147857666, 7.284163475036621], [11.634537696838379, -3.931627035140991, 3.887038469314575, -2.8694236278533936, -3.350072145462036, -5.562132835388184, -0.2493070363998413, -3.149024248123169, 10.920076370239258, -0.5485107898712158], [-2.6244564056396484, -9.272308349609375, 2.8152403831481934, 1.1114850044250488, -5.067164421081543, -4.324715614318848, 2.259734869003296, 10.226821899414062, 6.709073066711426, 5.215636253356934], [8.298758506774902, -2.5784740447998047, 0.7166876792907715, 0.3335554599761963, 3.0772931575775146, 0.8890725374221802, 6.157891273498535, -1.1037613153457642, -2.0949532985687256, -0.21016913652420044]], [[3.6268088817596436, 1.6467303037643433, -2.775477409362793, 3.512632369995117, -2.610487699508667, 1.6856930255889893, 0.11025130748748779, -0.8515074253082275, 3.382430076599121, -2.4342548847198486], [-7.05072021484375, -0.3707849979400635, -5.632901191711426, 5.043329238891602, 6.865029335021973, 4.931166648864746, 5.804108619689941, -0.6051623821258545, 4.156488418579102, 5.823215484619141], [-2.0617921352386475, 14.488234519958496, 13.043121337890625, 5.661471366882324, -5.79531192779541, -1.8218793869018555, -0.7771990299224854, -2.421753406524658, 5.822437286376953, 0.8184773921966553], [-0.6077849864959717, 3.4028453826904297, -16.40517234802246, 12.299433708190918, 9.598057746887207, 3.2114548683166504, -8.142251014709473, 0.16225585341453552, -5.459469795227051, 6.496912002563477], [-1.4511196613311768, -10.753203392028809, -3.9182121753692627, 10.035250663757324, -6.004203796386719, -11.660582542419434, -5.413904190063477, -3.939129590988159, 1.437072992324829, 0.06129467487335205], [4.687373161315918, -2.1408164501190186, 0.326619029045105, -1.3351945877075195, 2.9971413612365723, -1.7496788501739502, -4.754280090332031, 0.2680065631866455, 7.173888206481934, -1.1999233961105347], [-0.3728468418121338, -3.2927932739257812, -1.668860912322998, -4.176283836364746, 4.047055244445801, 5.678095817565918, 6.474268913269043, 3.583808422088623, -6.470794677734375, 14.386194229125977], [2.950681686401367, 13.51834774017334, -1.6061253547668457, 4.244509696960449, 0.7743275165557861, -13.964605331420898, -14.932989120483398, -1.435955286026001, -1.7227554321289062, -3.8037354946136475], [-5.887218475341797, -6.875885009765625, 5.964511871337891, 8.48865032196045, -2.017423391342163, -7.821824073791504, 0.2557070255279541, 2.841826915740967, 1.3555853366851807, 0.4080340266227722]], [[3.3139870166778564, 4.989319801330566, 4.501522064208984, -4.985773086547852, -0.4789748191833496, 6.156174659729004, -5.623822212219238, -4.139066696166992, 0.11002123355865479, 2.0698397159576416], [-1.7807791233062744, 3.303487539291382, 7.572134017944336, -6.345925331115723, -2.1250662803649902, 0.6141034960746765, -3.222322702407837, -9.213118553161621, 4.340061187744141, 1.8234469890594482], [-5.0480146408081055, -3.413682222366333, 6.627667427062988, 2.9978885650634766, -9.917438507080078, -13.37122631072998, 4.700967788696289, 4.490837097167969, -7.077801704406738, 11.630212783813477], [0.6678913831710815, -0.09489822387695312, 5.085424423217773, 1.8499289751052856, -4.528131484985352, -13.064522743225098, -4.324892044067383, 2.2910566329956055, -3.224815845489502, -7.4991350173950195], [-1.7572543621063232, 1.63797926902771, -11.0870943069458, -4.709198951721191, 5.781095504760742, -3.2927513122558594, -1.6573116779327393, 7.131339073181152, -3.870882272720337, -1.8717703819274902], [6.119622230529785, -1.6180329322814941, -0.03351259231567383, -14.642265319824219, -9.560251235961914, 7.528250694274902, 1.5978761911392212, -0.825659990310669, -0.38086938858032227, 0.4978184103965759], [2.0912208557128906, 4.175954341888428, -0.5323927402496338, 4.029120445251465, -4.292638778686523, 9.79151439666748, -4.510577201843262, 4.866678237915039, -3.099435806274414, -6.136239051818848], [-1.4070172309875488, -6.633542060852051, 12.901071548461914, -9.916886329650879, -1.857459545135498, 5.233412742614746, 6.759748458862305, 13.31435489654541, 3.7145090103149414, -1.7492966651916504], [0.1352766752243042, 10.039607048034668, 5.968634605407715, -7.355196952819824, -3.095876693725586, -6.607017517089844, 5.88050651550293, -1.139403223991394, -2.270936965942383, 4.85538387298584]], [[-0.2864266633987427, 3.413403034210205, 0.590609073638916, -1.8794782161712646, 0.022635221481323242, -3.7826011180877686, -3.711228609085083, 6.104622840881348, -1.2957007884979248, -2.0820720195770264], [2.263950824737549, 5.686820030212402, 2.3257699012756348, 7.100546836853027, -12.672341346740723, -1.9235711097717285, 8.255807876586914, 1.840061902999878, -13.167162895202637, 3.7712128162384033], [0.506697416305542, 3.246669054031372, -0.6840835809707642, -5.475214004516602, 8.342315673828125, 0.9585651159286499, 2.830155611038208, -4.131368637084961, 3.8731582164764404, -3.5186049938201904], [5.326380729675293, -11.79059886932373, -4.563240051269531, 7.49852180480957, 6.356584548950195, -1.965092420578003, -3.071213722229004, 9.567131042480469, 1.5214393138885498, 3.8187882900238037], [1.5042179822921753, 2.933905839920044, 5.782491683959961, -4.2374467849731445, 2.316046714782715, 3.886329174041748, -6.957449913024902, 4.0523200035095215, 2.004326820373535, -1.9242351055145264], [8.155526161193848, -7.708794593811035, 7.925127983093262, 5.055153846740723, -5.545876502990723, 5.268498420715332, 0.7622077465057373, 6.674358367919922, -1.4722182750701904, 1.824730396270752], [-11.092460632324219, 2.067589282989502, -3.7123944759368896, -8.255393981933594, 3.0607693195343018, 6.09503173828125, 9.03986644744873, -11.232680320739746, -0.6825432777404785, -4.826186180114746], [4.546079635620117, -13.86011028289795, 3.028531074523926, 5.812533378601074, 5.850573539733887, -1.0587916374206543, -0.597435712814331, 0.9647557139396667, 0.6725345849990845, -0.5375211238861084], [-3.5290415287017822, -2.2465085983276367, -0.35432541370391846, -0.8939237594604492, 6.649727821350098, 3.6331732273101807, 4.551662445068359, 2.6259965896606445, 0.5478578805923462, -4.032900810241699]]], [[[1.9889439344406128, 2.0354135036468506, 3.843745708465576, 4.093883991241455, 3.8680191040039062, 7.96143913269043, -8.889524459838867, 2.4182052612304688, 5.469849109649658, -4.4249796867370605], [-9.936166763305664, 4.198282241821289, 0.016842365264892578, 0.4990941882133484, 6.190547943115234, -1.0835009813308716, -11.462702751159668, 14.426311492919922, -2.4820122718811035, 2.888042449951172], [6.686299800872803, 4.12465238571167, 11.013215065002441, -1.7931660413742065, 8.019803047180176, -1.6889537572860718, -10.542984962463379, -13.232842445373535, 4.578031539916992, -1.1838794946670532], [2.855457067489624, 8.055516242980957, -7.023721218109131, -8.65306282043457, 7.128447532653809, -3.630260467529297, 10.76745319366455, 6.226007461547852, 0.2358243465423584, -0.26512134075164795], [4.564479827880859, -10.072765350341797, -3.1066627502441406, 15.063880920410156, 9.977872848510742, 5.986661434173584, 10.296305656433105, -6.97092866897583, 6.0505757331848145, 7.729592323303223], [2.748015880584717, -0.5291119813919067, 1.244213581085205, 1.4104058742523193, -1.2550095319747925, -4.112054347991943, 1.461932897567749, -3.0382633209228516, 4.083725452423096, 11.194531440734863], [7.2748589515686035, -4.197532653808594, 3.852083683013916, -15.35162353515625, 2.451777935028076, -1.5949047803878784, -4.133820056915283, 4.381178855895996, 0.3198893666267395, -7.869528293609619], [5.560846328735352, 5.415429592132568, -8.08543872833252, 1.4275763034820557, 3.824234962463379, 1.4259593486785889, 1.147331714630127, 14.356019973754883, -5.7629828453063965, -4.916958808898926], [-4.98889684677124, 5.069418907165527, -3.77610445022583, -2.59169864654541, 7.033421993255615, -6.124325275421143, 0.6475815773010254, 4.908468246459961, -8.248644828796387, -1.0223358869552612]], [[2.27496075630188, 8.254473686218262, -5.50885009765625, -2.8645834922790527, 7.314732551574707, 1.0989758968353271, 13.716464042663574, -6.423205375671387, 4.524876117706299, 11.54313850402832], [-1.3917030096054077, -1.762468695640564, -6.163695812225342, 10.139219284057617, 7.513397693634033, 1.1663360595703125, 6.154256820678711, -1.8709782361984253, -9.466510772705078, -0.7491816282272339], [5.853992462158203, 8.313862800598145, -5.597087860107422, 3.7803239822387695, 6.704239368438721, 5.314723968505859, -3.4979629516601562, 10.368571281433105, 10.957050323486328, 12.563403129577637], [10.170884132385254, -3.602306842803955, 2.155217170715332, 2.8759751319885254, -3.887913703918457, 7.231654167175293, 17.366121292114258, -3.020577907562256, -1.0655368566513062, -1.7062855958938599], [-0.34121382236480713, 18.706682205200195, 2.895009994506836, 1.7655620574951172, 8.39774227142334, -0.6022688150405884, 14.634465217590332, 1.4567793607711792, -10.912459373474121, 5.2602081298828125], [3.975335121154785, 14.157488822937012, -5.718217849731445, -23.971881866455078, 0.1694730520248413, 2.5709261894226074, -0.2252594232559204, -0.8017669916152954, -13.24891185760498, -1.4213045835494995], [-4.8767900466918945, 2.7059950828552246, 2.3467493057250977, 0.8137609958648682, -0.5229965448379517, -9.446090698242188, 7.853979110717773, -4.925062656402588, -3.577181816101074, -6.3972320556640625], [-0.29778552055358887, 4.408061504364014, -0.23634076118469238, 17.98089599609375, 5.938612461090088, -5.0957770347595215, -2.2485551834106445, -8.25414752960205, 8.707303047180176, 0.01470184326171875], [-10.385787963867188, 0.7612295746803284, 14.919580459594727, -4.5499467849731445, -3.000112533569336, 9.875528335571289, -9.967409133911133, -0.8291357755661011, 8.410452842712402, 20.058551788330078]], [[-5.790241718292236, -4.721248626708984, 9.623706817626953, -1.1524077653884888, 1.0014607906341553, -3.1155943870544434, 1.0425747632980347, 5.912992477416992, 7.258260250091553, 0.5102287530899048], [6.447033405303955, -3.671116828918457, 7.980031967163086, 13.229333877563477, 9.079512596130371, 0.13911020755767822, 6.331226825714111, -0.22749292850494385, 8.023361206054688, 1.6378865242004395], [4.139162540435791, 4.47289514541626, -2.3033313751220703, -0.31977975368499756, 9.360143661499023, 3.058756113052368, 3.8585023880004883, -2.0761337280273438, 9.697795867919922, 6.06110143661499], [0.02665388584136963, -0.7303100824356079, -2.3495383262634277, -3.6380667686462402, -11.410066604614258, -0.22510778903961182, 4.212344169616699, -5.624607086181641, 6.381966590881348, -5.437071800231934], [-5.542170524597168, 3.222470283508301, 1.1754969358444214, -4.74751615524292, -0.830105185508728, -0.7750296592712402, 10.236442565917969, 7.255644798278809, -3.0875320434570312, -1.5019606351852417], [1.9473001956939697, -0.012580156326293945, -4.4911675453186035, 11.428925514221191, -10.366463661193848, 11.653905868530273, -5.676713943481445, -10.513922691345215, 19.862228393554688, -10.900017738342285], [-1.1110275983810425, -3.316864490509033, -0.5514577627182007, 6.894136428833008, -8.360697746276855, -2.4302310943603516, -5.634178161621094, -13.211977005004883, -3.918947219848633, 3.936863899230957], [6.25299596786499, 7.5913214683532715, 2.013679027557373, 5.749845027923584, -15.127458572387695, 1.309858798980713, 5.227536678314209, 3.6316323280334473, 14.143632888793945, 6.194870471954346], [-3.068093776702881, 2.978015899658203, -5.811030864715576, 3.0559868812561035, 0.5004885196685791, 1.9022953510284424, 4.274781703948975, 3.7313876152038574, 4.148183822631836, 3.7227349281311035]], [[1.8547158241271973, 5.943460464477539, -6.009945392608643, -7.4563117027282715, -3.9746756553649902, -2.2353906631469727, 6.976634979248047, -5.016648769378662, -6.834599018096924, -0.6196179389953613], [0.9852503538131714, 4.5252885818481445, 8.404304504394531, -8.81218147277832, 10.20535659790039, -5.9303483963012695, -7.31576681137085, -4.028546333312988, 3.8988842964172363, 6.515454292297363], [-5.813317775726318, 8.528700828552246, 1.9572173357009888, -16.729877471923828, 0.23254770040512085, -0.404180645942688, 6.421947002410889, 0.511649489402771, -2.2475714683532715, 2.0445504188537598], [-4.266722202301025, -7.698791027069092, 1.855826497077942, -6.031301498413086, 8.145904541015625, -8.239898681640625, -2.5700230598449707, -0.903976559638977, -4.361969470977783, -0.0897977352142334], [2.6847004890441895, -0.9798141717910767, 6.0881757736206055, -1.7628244161605835, 10.61538028717041, 0.29018354415893555, -1.5929335355758667, -1.699198842048645, 10.443096160888672, 1.0413366556167603], [0.4121363162994385, 9.115784645080566, -1.1401206254959106, -1.104544758796692, -5.225484371185303, -17.469289779663086, -0.22108018398284912, -12.043639183044434, -8.924362182617188, 15.232646942138672], [2.8867459297180176, 9.2179594039917, -1.5349828004837036, 3.836263656616211, -3.4177327156066895, -11.690474510192871, 2.7110161781311035, 3.7625913619995117, -2.950590133666992, 7.171087265014648], [2.2520432472229004, 8.686197280883789, -3.3366007804870605, 5.868045806884766, 13.653018951416016, -10.87803840637207, 1.4660472869873047, 7.506993293762207, 11.487510681152344, -10.337409019470215], [0.9395301938056946, -11.16662883758545, 0.1644284725189209, 11.112630844116211, 8.141961097717285, -5.949447154998779, -1.038223385810852, 0.7599107623100281, 4.21099328994751, 15.992746353149414]], [[-6.883039951324463, -14.478638648986816, 0.825932502746582, 0.6484566926956177, 13.303531646728516, 5.246699333190918, 0.179082989692688, -8.491571426391602, 9.409147262573242, -0.8779586553573608], [-4.4538254737854, 10.81852912902832, 12.162002563476562, 16.70929527282715, -4.579290390014648, 2.048515558242798, -10.339975357055664, -7.712087154388428, -9.228963851928711, 11.317536354064941], [4.370513439178467, 16.23942756652832, -3.934436798095703, 2.597567081451416, 10.398571014404297, -4.931625843048096, -1.5291184186935425, -9.734807968139648, -10.791229248046875, -1.153361439704895], [-0.09724819660186768, 10.102206230163574, 2.8333497047424316, 9.890950202941895, 8.549519538879395, -7.86867094039917, -5.190171718597412, 5.4189605712890625, 3.638267993927002, -6.394871711730957], [7.378608226776123, 3.3233084678649902, 8.393416404724121, -3.2662391662597656, -6.381265163421631, -7.946950435638428, -2.111186981201172, -3.825676441192627, 11.20515251159668, 9.693347930908203], [-5.1814188957214355, 4.585563659667969, -9.294425010681152, -7.845808506011963, -5.367898941040039, 0.3990861177444458, 8.689166069030762, 4.682936191558838, -9.923053741455078, -6.004401683807373], [-4.28312349319458, -0.9101811647415161, -2.794769287109375, -4.266213893890381, 17.905012130737305, 6.312243938446045, -1.0482863187789917, -9.237739562988281, 5.263778209686279, -1.3237205743789673], [-2.3720006942749023, -5.05959939956665, 7.859269618988037, 11.703009605407715, -13.175390243530273, 5.741520404815674, 2.9094114303588867, 0.9265346527099609, 3.5716123580932617, -8.280926704406738], [-6.315568923950195, 1.4376153945922852, 4.042314052581787, 2.7992300987243652, -6.189638614654541, 10.731171607971191, 12.236710548400879, -9.524388313293457, 0.49036645889282227, -6.475089073181152]], [[3.0951895713806152, 7.242437362670898, 4.37935209274292, 11.396645545959473, -1.3914376497268677, -0.5809212923049927, -4.865027904510498, 3.8015804290771484, 1.8114879131317139, 5.455708026885986], [5.713114261627197, -5.6356425285339355, 6.601735591888428, -5.559447765350342, -5.418814182281494, -3.716749668121338, 16.743900299072266, 2.9556679725646973, 8.02047061920166, -8.411417007446289], [5.805388450622559, 5.452056884765625, 10.909615516662598, 3.600348472595215, -8.235418319702148, 5.64884614944458, 8.281378746032715, 10.240028381347656, -6.216508865356445, -1.4950882196426392], [-3.4675378799438477, 4.908784866333008, 14.66322135925293, -1.3158334493637085, 2.9714155197143555, 2.6374387741088867, -0.05983245372772217, 0.3318755626678467, 1.817995309829712, -0.636519193649292], [1.7193454504013062, 0.8358649015426636, 0.1654268503189087, -6.423868656158447, -17.18024444580078, 12.36164379119873, 0.16428983211517334, 4.947153568267822, -1.2216328382492065, -5.19709587097168], [-4.045391082763672, 1.3409098386764526, -9.622820854187012, -1.104347825050354, -5.0684003829956055, -8.129801750183105, 11.492280960083008, -5.528051376342773, 9.655147552490234, -4.419951915740967], [5.30474853515625, -9.315924644470215, 14.584702491760254, 2.1474661827087402, -4.58003568649292, 9.660451889038086, 2.349973678588867, 11.642694473266602, 4.2789154052734375, 21.296817779541016], [5.763065814971924, 20.664411544799805, -3.9862027168273926, -9.444965362548828, 2.7321369647979736, -1.5252827405929565, -5.019221782684326, 0.9533600807189941, -7.781871318817139, -2.1826930046081543], [1.130218267440796, 1.9095584154129028, 2.336505889892578, -21.33124542236328, -2.3215551376342773, -4.802768230438232, -1.7082983255386353, -5.371577739715576, 3.2772207260131836, 10.035656929016113]], [[10.023880958557129, 1.1656677722930908, -7.009301662445068, -3.419924736022949, 5.263946533203125, -3.740780830383301, -0.670326828956604, 4.189025402069092, 4.508883476257324, -8.102264404296875], [10.582706451416016, -4.260653018951416, -15.920293807983398, -2.4059057235717773, 2.370664119720459, 15.679332733154297, 0.1364363431930542, -11.886006355285645, 5.017655849456787, 3.446485996246338], [2.238964080810547, -0.9672476053237915, -2.963895797729492, -8.724769592285156, 9.53715991973877, 8.24069881439209, -0.49138522148132324, -10.847070693969727, 11.85515308380127, 4.099297523498535], [-0.07848513126373291, -9.2852144241333, 8.927538871765137, 6.648183345794678, -2.8376097679138184, -0.13985979557037354, -0.4725710153579712, -1.854606032371521, 6.898910045623779, 3.187816858291626], [-7.1223931312561035, -5.288547039031982, -6.231545925140381, 8.540972709655762, 9.557738304138184, -19.490835189819336, 3.1099534034729004, 2.79697847366333, 12.789397239685059, -4.609952449798584], [1.1794439554214478, -18.932138442993164, 6.763358116149902, -5.972365856170654, 0.550378680229187, 2.747209072113037, -4.273087978363037, 6.229494571685791, 2.1885480880737305, 3.9971981048583984], [3.910407066345215, 2.6044416427612305, -3.0880908966064453, 1.5412952899932861, 11.007231712341309, 7.048266410827637, -4.389911651611328, 1.2814595699310303, -5.836265563964844, -3.1850390434265137], [4.75319766998291, -7.609995365142822, 11.752166748046875, -0.980756402015686, 12.765867233276367, 1.0651941299438477, -7.399255275726318, -3.594879150390625, -0.9889975786209106, 3.9783401489257812], [-1.0092211961746216, -0.16461074352264404, 0.27027642726898193, 1.4552415609359741, 11.285691261291504, -0.025084853172302246, -2.4402856826782227, -8.740718841552734, 0.4207136034965515, -2.0346593856811523]]], [[[-4.914750576019287, 1.307842493057251, 2.6848347187042236, 0.6991981267929077, -5.253252029418945, -11.619001388549805, 4.850582122802734, -0.05509629845619202, 5.73948860168457, 3.6417582035064697], [1.774823546409607, 3.7120039463043213, 3.631303548812866, -1.4375113248825073, -10.459519386291504, 1.3977442979812622, 12.1664400100708, -1.4984468221664429, -8.288705825805664, -3.1989386081695557], [8.225403785705566, 7.45936918258667, -1.6200357675552368, -0.6073998212814331, 2.8777542114257812, 10.362932205200195, 1.98857843875885, 2.3136186599731445, -0.6207396984100342, 1.2381447553634644], [-3.333977460861206, -12.211490631103516, 2.495863199234009, 0.6229665279388428, -4.489908218383789, -4.2843523025512695, -5.973781108856201, -2.1518049240112305, -4.8355393409729, -3.3781538009643555], [-3.592017412185669, 1.541683316230774, 2.6538379192352295, -7.623762607574463, 1.2889281511306763, -1.6635102033615112, -5.319063663482666, 5.065488338470459, 3.7184102535247803, 2.388085126876831], [-8.68099594116211, 2.0115907192230225, -6.528476715087891, -3.827261209487915, -3.9276721477508545, -8.340415954589844, -5.964089393615723, 1.0879913568496704, -10.306218147277832, -4.92083215713501], [-1.2917414903640747, 4.3879194259643555, 1.78899085521698, 7.69559907913208, 3.988210439682007, -3.9036753177642822, 5.721837043762207, 1.9805384874343872, -7.691310405731201, -0.4036198556423187], [3.640521287918091, -0.3951303958892822, -5.7826104164123535, 3.4152395725250244, -2.038254737854004, -2.999387502670288, 3.9051544666290283, -9.266100883483887, -0.3157251477241516, -0.11635193228721619], [-5.338197708129883, -7.011995792388916, 3.817784070968628, 0.048163264989852905, -5.555540561676025, 7.4641547203063965, 2.453031301498413, -6.965342998504639, 0.8360601663589478, 4.71249532699585]], [[5.352168560028076, 10.0198974609375, -0.1752784550189972, 0.6944714784622192, 0.36861446499824524, -2.394078493118286, -10.93004035949707, 0.7973242998123169, 2.951486349105835, -1.1268036365509033], [-3.8371896743774414, -12.668668746948242, 0.8287729024887085, -0.59132981300354, -9.027822494506836, 2.922236680984497, 9.003988265991211, 7.367467403411865, -6.795233726501465, -11.973026275634766], [-3.572328805923462, 1.9286555051803589, -4.314798831939697, -13.607111930847168, 6.330803394317627, -5.5544843673706055, -8.26373291015625, -8.40139102935791, 1.3456753492355347, -0.6714520454406738], [5.332658290863037, 1.3376734256744385, -3.54762864112854, 3.845123052597046, 0.63800048828125, -3.51188588142395, 5.817399501800537, -11.09414291381836, -5.887626647949219, 0.8370522260665894], [-7.807313919067383, 0.8482322692871094, 4.967126846313477, 1.6472951173782349, 4.309385776519775, 1.4066005945205688, -3.4538464546203613, -2.341181755065918, 0.006695687770843506, -2.434361696243286], [5.525250434875488, -0.41662612557411194, -13.378044128417969, 1.0410802364349365, -8.207432746887207, -3.3075344562530518, -1.4979336261749268, 4.107476234436035, 10.56625747680664, 1.9596470594406128], [-7.362428188323975, -15.3858060836792, 3.082294225692749, -1.6931450366973877, -9.63024616241455, 2.2057173252105713, -6.268452167510986, -5.977046012878418, -12.654144287109375, 0.3020627796649933], [10.83875846862793, 8.877330780029297, 11.47722339630127, 0.12240448594093323, 15.822540283203125, 0.8153711557388306, 0.532920241355896, 14.286029815673828, 4.7492852210998535, 2.4051084518432617], [-0.07218345999717712, -4.61159086227417, 1.6856211423873901, -6.161816596984863, -1.63534677028656, -6.952328681945801, -0.8799139261245728, -4.4528727531433105, -12.812695503234863, -8.565964698791504]], [[-5.0113372802734375, 5.110629081726074, -3.8933756351470947, -1.1890852451324463, -7.093049049377441, 1.7410298585891724, -5.938072204589844, 4.625836372375488, 2.2676239013671875, 3.739295244216919], [2.733100175857544, -9.649279594421387, 1.0860956907272339, 7.805058002471924, -11.520537376403809, 3.1298232078552246, 4.342238426208496, -5.351972579956055, -16.197311401367188, -10.154267311096191], [-4.710513591766357, 1.6370126008987427, 7.930979251861572, -3.32812237739563, 6.322902202606201, 19.470651626586914, 1.840821623802185, -2.564307689666748, -4.158530235290527, 3.191462278366089], [7.345002174377441, 2.3885691165924072, -0.6802270412445068, -1.5995168685913086, -5.939828395843506, 0.27603432536125183, -6.152845859527588, -1.4091227054595947, -5.538261413574219, 1.6071728467941284], [-6.761806964874268, -8.70918083190918, 6.811895847320557, -10.554366111755371, -0.8590689897537231, -11.138030052185059, -3.784914255142212, 2.815681219100952, -9.48901653289795, -0.7998520135879517], [-1.9306597709655762, -4.253887176513672, -3.869978189468384, 0.35632088780403137, 16.054607391357422, -11.653606414794922, 6.843029499053955, 0.22994080185890198, -4.460943222045898, 8.10218334197998], [-5.9637370109558105, 0.12662169337272644, 16.241058349609375, -4.5624213218688965, 2.2609140872955322, -6.25070858001709, -0.5356577634811401, 8.5943021774292, -0.23146796226501465, 1.9319881200790405], [-4.478972911834717, 8.013331413269043, 4.52337646484375, -9.067656517028809, 3.4147369861602783, -3.668865919113159, 4.745899677276611, -2.9982941150665283, -12.652400016784668, -5.095163822174072], [-3.2786948680877686, -2.718228578567505, -5.863926410675049, -1.9610167741775513, 7.467473983764648, 10.51242446899414, 2.6946609020233154, -5.776761054992676, 7.161713123321533, 4.694735527038574]], [[1.3868168592453003, 5.2010955810546875, -7.342700481414795, -8.412904739379883, 0.3771495521068573, -4.906675815582275, 1.087385654449463, 1.2771092653274536, 2.298449993133545, -0.18416902422904968], [1.682017207145691, -11.429015159606934, -1.2798922061920166, 14.059269905090332, -4.831799030303955, -2.1021862030029297, 2.5452396869659424, -0.6348707675933838, 8.271879196166992, 1.6639307737350464], [-7.216305255889893, -7.042031288146973, 0.33399471640586853, 1.2013427019119263, 3.6472527980804443, -3.265864133834839, 4.574699401855469, -5.980743885040283, -17.219160079956055, -4.9814982414245605], [-8.4185209274292, -3.4681856632232666, -8.274463653564453, 12.294646263122559, 9.240460395812988, -3.8574392795562744, 9.85255241394043, -4.0765275955200195, 3.246025800704956, 2.3185181617736816], [5.759889602661133, 1.7451249361038208, -0.7817696332931519, 1.3188058137893677, -0.8812057971954346, -7.4502153396606445, 1.2874504327774048, -4.816555023193359, 5.130990982055664, -1.6023492813110352], [-1.0944586992263794, 1.1421152353286743, -0.28551915287971497, 5.5850830078125, -10.683156967163086, -1.8847469091415405, 6.8950910568237305, 3.507411003112793, 2.9009759426116943, -8.94959545135498], [-11.529662132263184, -7.397080898284912, -4.924571514129639, -7.208399295806885, -6.097189903259277, 7.456272125244141, -13.214750289916992, -4.441605091094971, 8.550409317016602, 2.3197665214538574], [4.81072473526001, 8.615399360656738, 4.068136692047119, 2.2743194103240967, 2.0430612564086914, -1.9225422143936157, 0.2577754557132721, 10.839218139648438, -1.4863481521606445, -4.736760139465332], [-2.9735074043273926, -1.6645933389663696, -3.088155746459961, -2.364725112915039, -9.656166076660156, -1.5633611679077148, -0.1514059603214264, -4.305930137634277, -0.33286765217781067, 5.074551582336426]], [[3.229875326156616, 12.855631828308105, -4.616524696350098, -4.688476085662842, 3.969902753829956, -7.381636142730713, -3.723085880279541, 3.077439069747925, -6.66896915435791, 3.3171939849853516], [0.12851175665855408, -3.474940061569214, 4.752652168273926, 2.4996540546417236, -9.80274486541748, -14.769383430480957, 13.532306671142578, 2.3050220012664795, 0.7441616058349609, -3.1068010330200195], [10.731030464172363, 6.445003986358643, 6.075360298156738, -6.771168231964111, 13.021121978759766, 6.146512508392334, 2.096745014190674, -0.41588690876960754, 9.412240982055664, 4.283779621124268], [-5.216391086578369, -20.05697250366211, 4.908693313598633, -6.203797817230225, -5.788514137268066, -12.754271507263184, -11.085585594177246, -7.266289234161377, -0.7655891180038452, -6.204467296600342], [-2.268747568130493, -5.482068061828613, -5.858862400054932, -11.010921478271484, 5.488178253173828, 12.160029411315918, 6.7617268562316895, 5.9348578453063965, -0.5708574056625366, 7.618919849395752], [-8.664036750793457, -1.3832767009735107, 4.1334662437438965, -5.974259853363037, 5.454501152038574, -2.025913953781128, 1.4852162599563599, 8.606193542480469, -1.976019024848938, -7.582793712615967], [-5.71419095993042, 1.571828007698059, 1.548745036125183, -3.6308319568634033, -5.892585277557373, 1.8172627687454224, 5.688030242919922, -1.0332772731781006, -6.532050132751465, 4.43092679977417], [-1.5665305852890015, 6.680142402648926, -0.1737542450428009, 7.975081920623779, -0.6112467050552368, -16.93291664123535, -3.1393256187438965, 0.18612059950828552, -14.088479995727539, -1.7771445512771606], [-0.25873953104019165, 2.9386978149414062, -7.80150032043457, 1.4635101556777954, 7.137434482574463, -0.22585633397102356, 2.321051836013794, -6.808189868927002, 1.9305340051651, 12.10363483428955]], [[0.5274794101715088, 2.195798873901367, -3.194425106048584, -1.1897506713867188, 1.1103403568267822, 3.6499617099761963, 3.0259218215942383, -4.0621418952941895, -3.8594706058502197, -11.779522895812988], [4.828853607177734, -6.219170093536377, -9.478978157043457, -0.5955837965011597, -8.734808921813965, -9.265966415405273, -1.358298420906067, -6.074345588684082, -5.349837303161621, 1.3301531076431274], [-3.814378499984741, 5.392340660095215, 6.310507774353027, -12.58873176574707, -0.2284896969795227, -6.181564807891846, -2.489675760269165, 4.292635917663574, -9.924661636352539, -3.2156827449798584], [3.53434681892395, -3.8406898975372314, 1.8293920755386353, -1.1685346364974976, 11.171268463134766, 5.158555030822754, 2.2677228450775146, -3.1176295280456543, -10.160456657409668, -2.108722686767578], [-11.888952255249023, 0.2307462990283966, -3.5157036781311035, -9.706197738647461, -0.8772960901260376, -11.754033088684082, 2.5000572204589844, -12.937702178955078, -8.918939590454102, 3.7549941539764404], [8.227697372436523, -3.239427328109741, -10.416661262512207, -1.7504938840866089, 9.96350383758545, -4.630894184112549, -5.281483173370361, -14.513104438781738, -6.367702484130859, -2.798442840576172], [-8.214423179626465, -5.132771015167236, -1.6878770589828491, 12.429876327514648, -1.9196313619613647, -2.8354780673980713, 6.75205659866333, -10.324560165405273, 2.2467777729034424, -7.773421764373779], [6.026060104370117, 9.09592056274414, -5.948497295379639, -11.170262336730957, -6.187900543212891, 5.376440525054932, -3.7746922969818115, -0.998355507850647, 9.335692405700684, -13.911272048950195], [-8.891776084899902, -17.560672760009766, -2.8333756923675537, 6.559926986694336, -2.1686129570007324, 3.3558318614959717, 6.547725677490234, 3.0074849128723145, -5.850926399230957, -4.491157531738281]], [[3.6154963970184326, 0.3600073754787445, 3.8585546016693115, -0.2118360698223114, 2.2996468544006348, 5.340453147888184, 1.2739551067352295, -6.080229759216309, -7.719173431396484, 0.7535287141799927], [-6.062894821166992, -0.058812737464904785, -0.36848360300064087, 0.0715310275554657, -2.457446813583374, -1.3684921264648438, 1.7185486555099487, -2.6243209838867188, -11.1893892288208, -1.6293245553970337], [-1.7563066482543945, 0.36807867884635925, -0.7945148348808289, 1.989111065864563, -1.7553845643997192, 2.1367132663726807, 1.0658315420150757, -1.1928595304489136, -3.306065320968628, -1.7324961423873901], [6.467820644378662, -6.04620361328125, 2.36316180229187, 14.432294845581055, -1.009657859802246, -4.011299133300781, -6.035033226013184, -1.8930555582046509, -0.0968862771987915, 4.24808931350708], [0.6861909627914429, 2.666179895401001, 8.17212963104248, -1.9567235708236694, 2.38836932182312, 4.4224853515625, -1.0651224851608276, 4.915106773376465, -0.8990638256072998, -1.8050035238265991], [-7.99960470199585, -5.791938304901123, -0.026425451040267944, -15.233735084533691, 2.7257556915283203, 6.284997940063477, 0.27791962027549744, -2.7448222637176514, -8.108405113220215, -3.627333879470825], [-7.558636665344238, 6.788693904876709, -3.2162206172943115, -5.933136940002441, -0.8614398241043091, -14.213133811950684, -2.290531635284424, -6.478401184082031, -2.346388339996338, -1.6135812997817993], [-5.670711994171143, -4.885894298553467, -7.435983657836914, 12.608765602111816, 5.739354133605957, 5.496802806854248, -1.8690004348754883, 0.9917242527008057, 2.2923085689544678, 8.118906021118164], [0.5783532857894897, -1.9262381792068481, -1.2490999698638916, -3.905391216278076, -10.820642471313477, 0.6301679611206055, -5.7281060218811035, -1.5487836599349976, -10.904732704162598, 0.9745280742645264]]], [[[8.092076301574707, 5.931911468505859, -1.9296876192092896, -3.4641716480255127, 11.591021537780762, -6.638576984405518, -4.700535297393799, -5.006030082702637, 5.074347972869873, 1.4797561168670654], [-1.7747035026550293, -8.39314079284668, -4.720322132110596, -5.53257417678833, 1.7786028385162354, -3.6579654216766357, 18.939851760864258, 0.6560165286064148, -13.397546768188477, 0.30073651671409607], [-4.3923659324646, 8.697701454162598, -3.509709596633911, 7.337253093719482, 8.74410343170166, -5.318721294403076, -5.573462963104248, -5.085912704467773, 4.696547031402588, 10.663856506347656], [10.130820274353027, 1.9649806022644043, -8.068135261535645, 1.8334656953811646, 0.9724293947219849, 2.1289315223693848, 7.209242343902588, 2.359609603881836, -5.573395729064941, -0.09733041375875473], [-6.505300045013428, -0.9375004172325134, 8.026726722717285, 1.3600019216537476, 0.29731515049934387, 4.094329357147217, 0.30651360750198364, -4.902107238769531, -1.672700047492981, -5.886413097381592], [-1.111262321472168, 8.70335578918457, -5.726863861083984, 0.7225616574287415, 3.77836537361145, 1.019801139831543, 1.7455906867980957, 4.832513332366943, 2.165398359298706, 3.924751043319702], [-14.372696876525879, 6.343886852264404, -3.7576797008514404, -6.830105304718018, -3.2097067832946777, -10.21481990814209, 1.732344150543213, -6.715907573699951, -9.893611907958984, 2.274487018585205], [10.26815414428711, 2.0702145099639893, 5.008788585662842, 5.4106059074401855, -3.7733917236328125, 4.241356372833252, 9.008346557617188, -3.9682276248931885, 4.407201766967773, 2.775158405303955], [0.5815305113792419, -0.1929672807455063, 0.39300668239593506, 4.542901515960693, -7.653578758239746, -1.1438040733337402, -2.0835797786712646, -7.767144680023193, 1.6744515895843506, 2.8989572525024414]], [[1.1805647611618042, 5.083618640899658, -11.157331466674805, -1.0354161262512207, -7.415729999542236, 3.429382562637329, -6.870743274688721, 8.486004829406738, 14.198734283447266, -1.837472915649414], [10.331477165222168, -7.782961368560791, 0.9923689961433411, 13.431208610534668, -2.4757251739501953, 5.964898109436035, -8.221559524536133, -9.97131061553955, 1.7633960247039795, -13.597606658935547], [-6.055643558502197, -4.105108261108398, 0.6782099604606628, 3.6586246490478516, 0.24243097007274628, 2.789903163909912, 1.428894281387329, -10.246320724487305, 18.895402908325195, 9.311739921569824], [3.6072919368743896, 2.358264923095703, -6.090455532073975, 3.5932066440582275, -10.285996437072754, 3.450315475463867, -0.18987281620502472, -12.013900756835938, 11.370508193969727, -0.25403475761413574], [0.8008846640586853, -2.5426785945892334, -7.497372150421143, -0.8620513081550598, -11.947888374328613, -12.104778289794922, 8.559000015258789, -3.4114346504211426, -3.3367977142333984, 0.020676350221037865], [7.628744125366211, -1.9131791591644287, -5.4267072677612305, 16.489606857299805, -3.2066380977630615, 2.4024691581726074, 1.585965871810913, -6.625226974487305, 0.6824706196784973, 3.590430498123169], [-3.4309887886047363, 6.091961860656738, 5.003653526306152, 5.157682418823242, -1.8726351261138916, 3.969125986099243, -8.182899475097656, 3.6508257389068604, 6.004176616668701, 13.854686737060547], [0.7663322687149048, -2.2115418910980225, 5.709552764892578, -8.801435470581055, -3.173941135406494, -2.114102602005005, 2.2635982036590576, -10.672837257385254, -3.1729888916015625, -7.042699813842773], [-2.23128604888916, -3.6531498432159424, -1.0855484008789062, -2.8607470989227295, 5.556122779846191, 1.8967891931533813, 1.5012586116790771, 5.8118157386779785, 7.946923732757568, -2.95503830909729]], [[-6.4750213623046875, -4.205975532531738, -15.812474250793457, -10.69007396697998, -6.599604606628418, 0.546206533908844, 3.4325194358825684, 4.868156433105469, -4.7748637199401855, -0.7668378949165344], [1.4439977407455444, -7.223396301269531, 4.533820629119873, -10.324885368347168, 6.529188632965088, 3.88027024269104, -11.93816089630127, -7.871112823486328, 5.955460548400879, 11.09298324584961], [8.08372688293457, -3.7730724811553955, 5.040347576141357, -21.95294761657715, 5.859534740447998, 4.389163494110107, -4.644895553588867, -10.421098709106445, -5.3790459632873535, 4.153544902801514], [-12.008578300476074, -6.870034694671631, 11.48422908782959, -2.5783541202545166, 8.81344223022461, -0.2628900706768036, -9.72374153137207, -1.3671867847442627, 1.8015668392181396, -3.4519052505493164], [3.1276800632476807, -1.6222267150878906, 3.207430839538574, 2.8859193325042725, -0.09277047961950302, 9.672114372253418, 6.088372230529785, 11.980238914489746, 2.463883876800537, 6.075187683105469], [2.512937307357788, 2.443247079849243, 1.6015878915786743, -4.6388702392578125, -3.7188737392425537, -5.6847100257873535, 7.6505327224731445, 6.335240364074707, -13.352989196777344, 9.793549537658691], [-5.102914810180664, 0.6324512362480164, -3.078669309616089, -6.244670391082764, 6.152099132537842, 6.145153999328613, -8.978382110595703, -4.307855129241943, 6.696262359619141, 2.9607982635498047], [11.655745506286621, 2.779637098312378, -0.48449018597602844, -6.7507781982421875, 5.258924961090088, 0.7667089104652405, 6.48625373840332, 14.29674243927002, -6.912024974822998, -2.1412036418914795], [0.863746702671051, -4.524944305419922, 1.6092075109481812, 4.9218244552612305, -4.04757022857666, -2.4938712120056152, -4.936084270477295, -6.270755290985107, 9.321561813354492, -0.6946435570716858]], [[6.596113681793213, 4.85959529876709, -8.709907531738281, 0.11514643579721451, -0.7968268394470215, 4.27324104309082, 4.77764368057251, -6.218761920928955, -0.047819361090660095, -0.6153278946876526], [-5.609958171844482, 6.113215923309326, 10.318209648132324, 18.045398712158203, -8.411263465881348, -0.7598486542701721, 5.650765895843506, -3.288520097732544, 7.35673189163208, -0.4744502007961273], [5.871992111206055, 13.77914047241211, -1.3936028480529785, -5.6263933181762695, -0.2496509850025177, 9.951424598693848, 11.195971488952637, -10.501676559448242, 11.846538543701172, 6.432980060577393], [-7.9267425537109375, 13.283827781677246, -0.540477454662323, 14.298189163208008, -11.264016151428223, -19.15921401977539, 0.1699787825345993, 2.0621912479400635, -3.359551429748535, -1.982738733291626], [-6.467832565307617, 4.641751289367676, -8.476922988891602, 7.202859878540039, 2.949373722076416, -0.5250079035758972, 3.946617603302002, 3.6703970432281494, 17.201257705688477, 0.6333993077278137], [7.679703235626221, -2.456913948059082, -0.005865057930350304, -10.945195198059082, -9.29439926147461, 4.057492256164551, 6.550418376922607, -3.5664689540863037, -1.9465081691741943, -10.457656860351562], [-0.9061905741691589, 3.632969379425049, 6.489633083343506, -0.9808892011642456, 1.8780128955841064, 13.934662818908691, 6.881696701049805, -10.023124694824219, 3.111987590789795, -4.076570987701416], [8.346561431884766, 4.597565174102783, -4.386995315551758, 7.362485408782959, 3.572338342666626, -13.514192581176758, -0.3179437220096588, 0.6506271958351135, -6.8202128410339355, -2.4392545223236084], [-0.1988927572965622, -0.08153524249792099, -3.89762544631958, 5.003767490386963, -7.046234130859375, 3.46964693069458, 9.490933418273926, -10.211580276489258, 5.6675124168396, 4.510076999664307]], [[0.5482795834541321, 0.7661758065223694, -4.206589221954346, 5.994111061096191, 1.8633074760437012, 3.3081657886505127, 1.5436186790466309, 1.8914897441864014, 4.378646373748779, 0.7032493948936462], [0.34627386927604675, -0.5049210786819458, -4.021477699279785, -8.213102340698242, -9.398724555969238, -6.6090264320373535, 3.092074394226074, -8.001200675964355, -2.9543330669403076, 3.3137829303741455], [-5.96448278427124, -12.483651161193848, -3.8122119903564453, -10.846701622009277, -7.95330286026001, 0.6097286939620972, -7.181234359741211, -5.589638710021973, 2.262699604034424, -6.2288594245910645], [7.5399699211120605, 6.296571254730225, 4.064990997314453, 2.9360201358795166, -0.5095736384391785, 6.315308570861816, 3.8622591495513916, 3.631936550140381, -1.7699410915374756, 4.472225666046143], [-0.32835620641708374, 10.130879402160645, 11.16409683227539, 8.123804092407227, -6.979693412780762, -10.435528755187988, -1.9491393566131592, 3.0509531497955322, -2.36834979057312, -1.5195966958999634], [0.07746223360300064, 2.8641469478607178, -14.351078987121582, 3.7804598808288574, 7.357333183288574, 11.907739639282227, -5.328941822052002, -11.960240364074707, -10.776632308959961, -10.821258544921875], [-4.874526500701904, 6.7565388679504395, 3.0417819023132324, 5.836763858795166, -2.786210536956787, 13.090465545654297, 4.451419353485107, -2.8613452911376953, 8.245487213134766, 6.387715816497803], [5.364349365234375, 8.286349296569824, 1.9227533340454102, -10.0224027633667, -5.36824893951416, 5.070449352264404, 7.979903697967529, 11.650748252868652, -2.1353235244750977, 5.825016498565674], [-3.66190505027771, -6.065066814422607, -4.949500560760498, -2.7159156799316406, 3.907334089279175, -1.2407629489898682, 0.1531485766172409, 3.1736562252044678, -4.003015518188477, 4.285449504852295]], [[10.18641471862793, -3.0850069522857666, -2.407912254333496, 6.682440280914307, -0.46857380867004395, -2.3509795665740967, -4.942026615142822, 5.938715934753418, -4.4079155921936035, -5.612730026245117], [-3.7622122764587402, -6.002585411071777, 1.93129563331604, -4.313889980316162, 5.848735809326172, 16.851411819458008, 11.289605140686035, 3.9534878730773926, 2.007431983947754, 2.2604029178619385], [-0.6626578569412231, 5.7214250564575195, 2.367418050765991, -9.739297866821289, 13.179150581359863, 7.479910850524902, 4.019426345825195, -6.7750983238220215, -0.38497695326805115, 6.696600437164307], [-7.098501682281494, 0.013862887397408485, 0.3626682758331299, -2.4020297527313232, 12.000594139099121, 0.6297072768211365, 10.23996353149414, -6.773731708526611, 8.265499114990234, 9.953362464904785], [2.314828395843506, 10.001995086669922, -0.08334928005933762, 5.439957618713379, 7.396683216094971, -11.024558067321777, 0.4053122103214264, 6.707223415374756, 4.59588623046875, 5.303776264190674], [-3.7583084106445312, -8.562230110168457, 13.04347038269043, 8.280777931213379, -8.096830368041992, 7.07097864151001, -2.701044797897339, 0.6060697436332703, -1.2903308868408203, 1.2404742240905762], [2.709514617919922, 1.0326510667800903, -16.972768783569336, 1.4749037027359009, 2.968427896499634, 10.335460662841797, -8.199312210083008, -2.741673707962036, 1.7663521766662598, -3.6476917266845703], [3.0836212635040283, -7.08482551574707, 6.977838039398193, 7.144194602966309, 9.017308235168457, 17.709829330444336, -6.54749870300293, 1.734748125076294, -0.3638278543949127, -0.9768574833869934], [2.7024171352386475, 3.050666093826294, -5.883917331695557, -2.3477232456207275, -3.5308609008789062, 0.8107110857963562, -4.507290840148926, 5.341655254364014, -2.651115655899048, -5.230827808380127]], [[8.148387908935547, -6.123740196228027, -0.5149809122085571, 2.4588358402252197, 1.4027460813522339, 3.5507071018218994, -7.924862861633301, -6.52838659286499, 1.1360796689987183, 5.388321399688721], [3.2882394790649414, -6.446497917175293, 2.60154390335083, -6.1390252113342285, -10.105364799499512, -1.340192198753357, -10.399039268493652, 1.9562171697616577, 1.3991851806640625, 4.83870792388916], [0.1500888615846634, 6.623366355895996, 7.564948081970215, -8.204195022583008, 6.168665885925293, 4.719283103942871, 3.980107069015503, -3.1655383110046387, -1.5274443626403809, -1.6558771133422852], [-2.288912773132324, -10.266780853271484, 3.6190240383148193, 2.8487884998321533, -8.02354621887207, -6.685794353485107, -6.4471259117126465, 0.6758216023445129, -1.3077834844589233, 0.9889218211174011], [-6.719275951385498, -0.7165831923484802, -6.821870803833008, -5.729801654815674, 0.7968437075614929, -5.935741424560547, -6.955831527709961, -0.34321391582489014, -0.6234717965126038, -6.491968631744385], [-0.4249285161495209, 0.2200162708759308, 8.317557334899902, 2.44269061088562, 15.475956916809082, -12.653576850891113, -3.651557683944702, -6.0731048583984375, 2.622253894805908, 4.5030717849731445], [-6.899069786071777, 16.190000534057617, -2.2812702655792236, -12.531152725219727, 6.824115753173828, -17.860876083374023, 10.632355690002441, -10.884089469909668, 8.589120864868164, 2.966059684753418], [-2.8579840660095215, -0.025696152821183205, 2.4785196781158447, 7.3341064453125, 5.277890205383301, -2.94844126701355, 8.676884651184082, 1.998380184173584, 1.2202895879745483, 1.86385178565979], [-1.7014329433441162, -0.11608882993459702, 5.572253227233887, 2.5899441242218018, 0.7675856351852417, 2.7301526069641113, -0.7924566864967346, 0.2227945774793625, -3.5002527236938477, 1.2285406589508057]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[0.9573068022727966, 0.9775368571281433, 0.8590018153190613, 0.19726839661598206, -0.4337151050567627, -0.17449833452701569, -0.3872487246990204, -1.1181426048278809], [0.8419092893600464, 1.968459129333496, 0.6181021332740784, 0.5616270899772644, 0.9401994347572327, -0.014009521342813969, -0.7574827671051025, 0.6141722202301025], [0.4097226858139038, -1.4108316898345947, 1.049096703529358, -0.7918463349342346, 0.9244181513786316, 0.11809256672859192, -0.5093831419944763, -0.9023868441581726], [-0.8913450241088867, -1.903050422668457, -0.9664616584777832, 0.14316031336784363, -1.3665956258773804, -0.42221447825431824, 0.4133603274822235, -2.4276695251464844], [-1.1392827033996582, 1.040285587310791, -0.6485775113105774, -0.38399040699005127, 0.8668878078460693, -1.1804097890853882, -2.6991631984710693, 0.6362320780754089], [0.27453678846359253, -1.0568989515304565, -2.4216535091400146, 1.4366099834442139, -0.1294407695531845, 0.0497627928853035, 0.6023318767547607, 1.172268271446228], [-0.4232030510902405, -0.0400414876639843, 1.7341089248657227, -0.05673541501164436, 0.06340647488832474, 0.9496986865997314, -2.5355048179626465, 0.2659797966480255], [-0.9683730006217957, 0.08427606523036957, -0.29010921716690063, -0.48609036207199097, 0.6778541207313538, -0.531997561454773, -2.32395601272583, 1.6361981630325317]], [[-0.6494061350822449, -0.23916758596897125, 0.9588046073913574, 0.5220412015914917, -0.5864295363426208, -0.3439117670059204, 0.6989185214042664, -0.5962659120559692], [0.49474406242370605, 2.6163265705108643, -0.370500773191452, -2.428194284439087, -1.3609024286270142, 0.6181468367576599, -1.274989128112793, 0.5614119172096252], [0.6585406064987183, 1.5771280527114868, -1.0982526540756226, 1.7431774139404297, -1.529363751411438, 0.6275295615196228, 1.2844804525375366, 0.8070903420448303], [0.3111852705478668, 0.24605663120746613, -0.24233746528625488, -0.16226300597190857, 0.5288872122764587, 0.5798529386520386, 0.1177249625325203, -0.34869253635406494], [-0.5765216946601868, 0.005596677772700787, 0.3794407844543457, 0.9692389965057373, -0.853350818157196, 0.028652789071202278, -1.0704807043075562, 0.9942770600318909], [0.3687689006328583, -0.976706862449646, -1.9047307968139648, 0.9027083516120911, -0.9113925099372864, -0.6410226821899414, 1.7332744598388672, -0.5514804720878601], [-0.4371192455291748, 1.036794662475586, 0.2989693284034729, 1.1333638429641724, -0.6532132029533386, -0.6253604888916016, -0.7654910087585449, 1.8821603059768677], [-0.8932624459266663, -0.4585098922252655, 0.008837336674332619, 2.499718189239502, 0.2828422486782074, 0.24116986989974976, -1.7630915641784668, 0.3010311722755432]], [[-0.43469613790512085, -0.7055643796920776, -0.26415327191352844, 1.8949826955795288, -1.102344036102295, 0.09735990315675735, 1.2725530862808228, -0.703076183795929], [-1.9225471019744873, 0.6378545761108398, -0.4862716495990753, -0.7299171686172485, 1.5057034492492676, 0.07961864024400711, 1.1416279077529907, -0.428569495677948], [-0.9316014647483826, 1.597256064414978, 1.4928200244903564, -1.0261499881744385, -0.8135435581207275, 0.3844607174396515, -0.6162068843841553, 1.3017735481262207], [-0.8635827898979187, -0.9880411028862, -1.772209644317627, -0.5827480554580688, -0.8257375955581665, 0.013358662836253643, 0.8009006381034851, -0.08026918023824692], [2.3974599838256836, -0.8156939148902893, -0.442345529794693, 0.3106784224510193, 0.24211162328720093, 1.5151864290237427, 1.0618388652801514, 1.2466981410980225], [-0.3765254318714142, 0.5388237833976746, 0.402908056974411, -0.17082034051418304, -1.3185019493103027, 0.05728363245725632, 1.0532276630401611, -0.7531765699386597], [-0.17403605580329895, 0.2536143362522125, -0.3392748236656189, 0.8563727736473083, 1.4818000793457031, -0.6996344327926636, -0.6509300470352173, -2.4715373516082764], [-1.005864143371582, -0.02034689672291279, -0.8986450433731079, 0.9400944709777832, -0.5382935404777527, -0.42435401678085327, 0.05300808697938919, 1.5722119808197021]], [[1.1752506494522095, 0.08627625554800034, 0.027432719245553017, 1.0176563262939453, 0.9600756764411926, 0.022916346788406372, 0.2118155062198639, -1.2169520854949951], [0.18749584257602692, -0.8038721084594727, 1.864759922027588, -0.08729139715433121, 0.3213406801223755, 0.6381263732910156, 1.542862057685852, -1.0612622499465942], [1.194148302078247, -1.6977143287658691, 0.36160531640052795, 0.8361576795578003, -1.4900370836257935, -0.24896417558193207, -1.3913226127624512, -0.6614759564399719], [0.2933118939399719, 0.8901253342628479, -0.4770892262458801, -0.1438378542661667, -1.1844538450241089, 0.5382644534111023, -0.4662860929965973, 0.10066282749176025], [0.5513992309570312, -1.6709558963775635, -0.7584213018417358, -1.4042857885360718, 1.2144373655319214, 1.0104904174804688, -2.0841267108917236, 0.7231981754302979], [1.2066450119018555, -0.27090564370155334, -2.70102858543396, 1.0958466529846191, 0.8746693730354309, -0.5805957913398743, -0.7620220184326172, 1.2109323740005493], [0.20585905015468597, -0.19695641100406647, 0.2157033532857895, -1.8431984186172485, 1.1777973175048828, 0.6944831013679504, 0.6685329675674438, -0.7472633123397827], [-0.5205204486846924, 0.08632581681013107, -0.3022903800010681, 0.7810683846473694, 0.05616603419184685, -1.0604732036590576, -0.8047637343406677, 0.23876214027404785]], [[-0.704817533493042, -2.1479039192199707, 0.437939316034317, 0.6097338199615479, -1.9104583263397217, 0.3654819428920746, -0.9931632280349731, -1.325021505355835], [0.6587367057800293, -0.693535327911377, 0.2627487778663635, -1.0312682390213013, -0.1892162710428238, 0.3099629580974579, 1.167224407196045, -0.6292112469673157], [2.372995376586914, 0.3607313930988312, 0.7044907212257385, 1.4435501098632812, 1.5431404113769531, 0.12888562679290771, 0.6275327205657959, 0.21398061513900757], [-0.022275026887655258, -0.32372787594795227, -0.6497630476951599, -1.448175072669983, -0.5592217445373535, -0.3353319466114044, 0.5391497015953064, -0.24169112741947174], [1.1231350898742676, 0.7698197960853577, -0.9826779961585999, -0.6667818427085876, 0.7018977403640747, -1.3177913427352905, -0.6441076397895813, -1.0073879957199097], [1.366982340812683, -0.17378835380077362, -0.6223303079605103, 0.10943055897951126, 1.6529115438461304, -1.5274139642715454, 1.2256767749786377, 0.7323312759399414], [0.2069614678621292, -0.6268166899681091, 1.0838770866394043, -1.30106520652771, -0.4375074505805969, 0.9454474449157715, -0.7385927438735962, -0.36574387550354004], [0.9505068063735962, 1.6599632501602173, -0.6443474292755127, 0.42756348848342896, -0.1485641598701477, 0.8049112558364868, -0.431856632232666, 1.6520352363586426]], [[0.6202669739723206, -0.17503902316093445, 1.7637012004852295, -0.9477916955947876, -0.38769975304603577, 2.250349998474121, -1.3377636671066284, 0.2497912496328354], [-0.17194636166095734, -0.3791312575340271, -2.2066330909729004, -0.9332894682884216, -1.6220088005065918, -1.7179625034332275, -2.3280189037323, -0.3419792652130127], [0.6106951832771301, 0.2112167775630951, -0.9995719194412231, -1.2879829406738281, -0.3788527250289917, 1.934456706047058, -0.22445322573184967, -0.5522451996803284], [-0.02005827985703945, 0.14495475590229034, 0.21254666149616241, -1.410226583480835, -1.4181807041168213, -0.23998315632343292, 0.924734354019165, 0.25012847781181335], [0.2934882342815399, 1.7129400968551636, -1.3578754663467407, 0.4659017324447632, -0.14325891435146332, 1.083686113357544, -0.28654903173446655, -0.09095079451799393], [1.2225888967514038, 0.1432407796382904, -0.44849467277526855, -1.1524810791015625, -1.2054039239883423, 1.3700512647628784, 1.2041432857513428, -0.6585265398025513], [1.7231720685958862, 0.1343955099582672, -0.1655004620552063, 1.6127492189407349, 0.5558037757873535, -1.592374563217163, -0.3665272295475006, -0.4579731822013855], [-1.513256311416626, 1.6852481365203857, -0.6779320240020752, 0.5580978989601135, -0.833055317401886, 0.6007510423660278, 1.703766107559204, -0.005395069252699614]], [[0.7613546848297119, 1.0616339445114136, -1.3194105625152588, 1.012704849243164, -0.2422015368938446, -1.0682545900344849, -0.09739982336759567, 1.0618221759796143], [-1.3834636211395264, -0.6894869208335876, 0.4552314877510071, 0.14480991661548615, 1.8304641246795654, -1.6213791370391846, 1.5674045085906982, -1.345879316329956], [0.23592978715896606, 0.9324973225593567, 0.9762954115867615, -0.5761879086494446, -0.2908301055431366, 1.1018325090408325, 0.15340882539749146, -0.12920774519443512], [1.0670721530914307, -1.4185081720352173, -1.5507601499557495, 0.03980787470936775, -0.9643332958221436, 0.5496811270713806, 0.9156278967857361, -1.3666294813156128], [-0.7791489362716675, 0.1696540117263794, -0.2372729629278183, 0.925030529499054, 0.7434568405151367, 0.2630643844604492, 0.2799729108810425, -2.137009382247925], [0.5872085094451904, 0.2755489647388458, 0.42843660712242126, 0.22810006141662598, -0.19606229662895203, -0.4392046630382538, -0.8619016408920288, 0.8747067451477051], [-0.029419943690299988, -0.21813051402568817, 0.6031129956245422, -0.2800516188144684, 0.45208489894866943, -0.9866793155670166, -0.8208081722259521, -0.7000563144683838], [0.10165884345769882, -0.9101983308792114, -1.6514205932617188, -0.6486722230911255, -0.90534508228302, 1.1445016860961914, 0.025920992717146873, -0.44034484028816223]], [[-0.189665749669075, -0.570202648639679, 1.387877106666565, -1.7432340383529663, 0.11007140576839447, -1.0773639678955078, -0.2336946278810501, 0.9129096865653992], [0.39544299244880676, 0.3559601902961731, 1.756888508796692, -0.6025413870811462, 1.1600464582443237, -0.7071794271469116, -1.0379997491836548, 0.12225554138422012], [-0.000785421347245574, -1.5060704946517944, 0.8578230738639832, -0.8295950293540955, 0.16138048470020294, 0.021440068259835243, -0.48025810718536377, 0.6254069805145264], [1.7618316411972046, 1.038388729095459, 1.094940423965454, -2.3840525150299072, -0.2625579833984375, -0.8571861386299133, 0.6879544854164124, -0.13559140264987946], [0.2827167212963104, 0.9777677059173584, 1.1965123414993286, 0.06448899954557419, -1.0274357795715332, -1.2353556156158447, -0.04222206026315689, -1.523730754852295], [2.5061028003692627, 0.261909544467926, -0.46798887848854065, 1.4546983242034912, 0.5009410381317139, 1.4010034799575806, -1.763049840927124, -0.5816553235054016], [-1.559038758277893, -1.7035857439041138, -1.6760718822479248, -0.1600153148174286, -1.642688512802124, -0.2743339538574219, -0.41242533922195435, 1.2957454919815063], [0.9810296893119812, -0.5079570412635803, -0.08664459735155106, 1.2019548416137695, -2.0284922122955322, 2.0053813457489014, -0.8971356153488159, -2.8104357719421387]]]], "dtype": "float32"}, {"__nd__": [[[[[0.4715564250946045, -0.14827218651771545, -0.2339976280927658], [1.5088509321212769, -1.0043895244598389, 0.9197815656661987], [0.4382227957248688, 0.848770022392273, -0.3432616591453552]], [[0.27218955755233765, -0.10195630043745041, 0.978747546672821], [-0.3065640330314636, 0.6462984085083008, -0.12809737026691437], [-1.1012792587280273, 1.3538285493850708, -1.3123341798782349]], [[-1.2593199014663696, 0.31396254897117615, 0.452392578125], [0.13160833716392517, 1.29840886592865, -0.650629997253418], [1.0621761083602905, -1.5350613594055176, 0.6714267134666443]]]], [[[[-2.7964582443237305, 1.3104718923568726, 0.3497183620929718], [-0.7670484185218811, 0.746568500995636, 1.354693055152893], [1.5068844556808472, -0.5439556241035461, 0.024448363110423088]], [[-1.1612331867218018, 0.3160669505596161, -1.2209036350250244], [-0.1258445680141449, 2.839808702468872, -0.11862420290708542], [0.45292380452156067, -0.0474034920334816, -0.04342251271009445]], [[1.1686733961105347, -0.7625150084495544, -0.12073778361082077], [-1.5528627634048462, 0.597678542137146, -0.8088628649711609], [-0.9302616715431213, -0.36238420009613037, 1.2400317192077637]]]]], "dtype": "float32"}, {"__nd__": [0.8411188721656799, -0.3454612195491791], "dtype": "float32"}, 3, 2], "e": {"__nd__": [[[[1.4838802814483643, 0.6932659149169922, 0.3882935643196106, -0.3465455174446106], [0.35694167017936707, 0.060339391231536865, -0.609456479549408, -2.629694700241089], [-0.1370558738708496, -7.806960105895996, -5.279477119445801, 0.47669684886932373], [0.40303412079811096, 0.4240012466907501, -1.230884075164795, -1.2193779945373535]], [[2.4235963821411133, 0.03628414869308472, 0.8549106121063232, 0.06148737668991089], [0.10001260042190552, 10.152491569519043, -0.21244388818740845, 3.099834680557251], [3.69097900390625, -4.271490097045898, -1.1382958889007568, 3.049673557281494], [-0.1698264479637146, 1.1945964097976685, 0.5321756601333618, 1.110333800315857]], [[0.7802520394325256, 7.719580173492432, 7.2013630867004395, 1.113216519355774], [2.594151258468628, 6.068474769592285, -0.6981067061424255, 0.788401186466217], [-1.1204359531402588, 0.676182746887207, 6.603274822235107, 4.0549750328063965], [-0.8164035677909851, 3.027730703353882, 3.6435036659240723, 2.173213243484497]], [[0.9062238335609436, 2.367616891860962, 0.05513876676559448, 1.241176724433899], [0.1430944800376892, -0.804848849773407, 0.4373618960380554, 1.7829959392547607], [3.6151890754699707, 1.1986994743347168, -2.534395217895508, -0.18721216917037964], [0.6115602254867554, 0.33318084478378296, -0.20284432172775269, -0.4841601252555847]]], [[[0.8416295051574707, -1.3214958906173706, -0.3589579463005066, 0.6947039365768433], [-1.8838170766830444, 7.172337055206299, 1.8394471406936646, 4.031956195831299], [-0.9547545909881592, -0.21848493814468384, -1.910085916519165, -1.669716238975525], [-0.2285420000553131, 0.03293207287788391, 1.1329742670059204, 1.5667200088500977]], [[1.1148855686187744, -0.43224772810935974, -2.2229013442993164, -0.43032106757164], [2.5977184772491455, -5.355812072753906, 8.752941131591797, -3.069180965423584], [-3.72035813331604, -1.3945081233978271, 2.7405643463134766, -2.679126501083374], [0.6330581903457642, 0.6514347791671753, -0.03928282856941223, -2.7339518070220947]], [[0.5544775724411011, -3.180823802947998, -3.1799216270446777, -3.2167513370513916], [4.53680944442749, 2.2614428997039795, 15.198822975158691, 1.3647429943084717], [0.9086672067642212, -10.363299369812012, -0.8057053089141846, -1.8646799325942993], [1.8222144842147827, -8.26114559173584, -1.882989525794983, -5.473663330078125]], [[-0.35009822249412537, -2.002253532409668, 0.40072837471961975, 1.0301882028579712], [-0.16515764594078064, 2.3333969116210938, -5.550165176391602, -1.3713817596435547], [3.1102941036224365, -1.7281228303909302, -3.1688361167907715, 6.314284801483154], [-0.002377122640609741, 1.381819725036621, 7.641383647918701, 7.513804912567139]]]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(conv3d, json.loads(_TESTS), 'cuda', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")